# NanoScreen-AI analysis workflow

Public notebook for reproducing the NanoScreen-AI analysis workflow. 
This version provides the finalized computational workflow with repository-relative paths and English documentation.


# Data loading

Load the analytical dataset from the public repository data directory and define global analysis parameters.


In [ ]:
import os
import re
import hashlib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path

# Paths and global parameters
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
FIGURES_DIR = PROJECT_ROOT / "figures"
MODEL_DIR = PROJECT_ROOT / "models"

PATH_2024 = DATA_DIR / "dataset.csv"
SAVE_DIR = str(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Target delivery endpoint for binary classification
TARGET_DE = "DE_tumor"  # For example: DE_tumor / DE_liver / DE_spleen.

# Full feature columns in the dataset
# No, Type, MAT, TS, CT, TM, Shape, Size, Zeta Potential, Admin, DE_*
FEATURE_CANDIDATES = ["Type", "MAT", "TS", "CT", "TM", "Shape", "Size", "Zeta Potential", "Admin"]

# Group split to avoid formulation-level leakage
GROUP_COLS = ["Type", "MAT", "TS", "CT", "TM", "Shape", "Size", "Zeta Potential", "Admin"]

TEST_SIZE = 0.20
VAL_SIZE  = 0.20
RANDOM_STATE = 42

# Classification threshold based on the training-set quantile
QUANTILE_P = 0.75  # Sensitivity settings can use 0.70/0.80

# =========================
# Core switches used for comparator experiments
# =========================

# Whether to construct interaction features
USE_INTERACTIONS = False

# Whether to keep other DE_* columns as features
KEEP_OTHER_DE_AS_FEATURES = False

# Imbalance handling method applied only to the training set
# "none" / "random_oversample" / "class_weight"
IMBALANCE_METHOD = "random_oversample"

# Scaling mode:
# "standard" / "minmax" / "robust" / "none" / "auto"
SCALER_MODE = "standard"

# one-hot:Low-frequency category grouping
OHE_MIN_FREQUENCY = 2

# Whether to display EDA plots
SHOW_PLOTS = True


# Data preprocessing

Standardize categorical labels, clean numerical fields, and prepare the curated analytical table.


In [ ]:
# 1) Text cleaning and semantic mapping
BASE_CAT_COLS = ["Type", "MAT", "TS", "CT", "TM", "Shape", "Admin"]

TYPE_MAPPING = {
    "Inorganic": "INM",
    "Organic": "ONM",
    "INM": "INM",
    "ONM": "ONM",
    "Hybrid": "Hybrid",
    "Other": "Other",
    "Others": "Other",
}
TM_MAPPING = {
    "XH": "Xenograft Heterotopic",
    "AH": "Allograft Heterotopic",
    "XO": "Xenograft Orthotopic",
    "AO": "Allograft Orthotopic",
    "Xenograft Heterotopic": "Xenograft Heterotopic",
    "Allograft Heterotopic": "Allograft Heterotopic",
    "Xenograft Orthotopic": "Xenograft Orthotopic",
    "Allograft Orthotopic": "Allograft Orthotopic",
    "Other": "Other",
    "Others": "Other",
}
MAT_MAPPING = {
    "Dendrimers": "Dendrimer",
    "Liposomes": "Liposome",
    "Hydrogels": "Hydrogel",
    "Other": "Other",
    "Others": "Other",
}

def _strip_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]
    return df

def normalize_cat_text(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    s = s.str.replace("\u00a0", " ", regex=False)
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.strip()
    return s

def normalize_categorical_df(df: pd.DataFrame, cat_cols) -> pd.DataFrame:
    df = df.copy()
    for c in cat_cols:
        if c in df.columns:
            df[c] = normalize_cat_text(df[c])

    # Common normalization:cervix
    if "CT" in df.columns:
        df["CT"] = df["CT"].astype(str).str.replace(
            r"^\s*cervix\s*$", "Cervix",
            regex=True, flags=re.IGNORECASE
)

    # Others -> Other
    for c in ["CT", "Shape", "MAT", "Type", "TM", "Admin"]:
        if c in df.columns:
            df[c] = df[c].replace({"Others": "Other", "others": "Other"})

    return df

def apply_semantic_mapping(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Type" in df.columns:
        df["Type"] = df["Type"].replace(TYPE_MAPPING)
    if "TM" in df.columns:
        df["TM"] = df["TM"].replace(TM_MAPPING)
    if "MAT" in df.columns:
        df["MAT"] = df["MAT"].replace(MAT_MAPPING)
    return df

def clean_2024(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Remove the No column
    - Remove other DE_* columns except TARGET_DE by default to avoid leakage
    - Standardize categorical text and apply semantic mappings
    - Convert numeric columns to numeric dtype
    """
    df = _strip_cols(df)

    if "No" in df.columns:
        df = df.drop(columns=["No"], errors="ignore")

    #  DE
    de_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("DE_")]
    if TARGET_DE not in df.columns:
        raise ValueError(f" {TARGET_DE}.DE:{de_cols}")

    if not KEEP_OTHER_DE_AS_FEATURES:
        drop_de = [c for c in de_cols if c != TARGET_DE]
        df = df.drop(columns=drop_de, errors="ignore")

    df = normalize_categorical_df(df, BASE_CAT_COLS)
    df = apply_semantic_mapping(df)

    # Numeric columns:Size(log10) / Zeta/ Admin
    for c in ["Size", "Zeta Potential", "Admin"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # numeric
    df[TARGET_DE] = pd.to_numeric(df[TARGET_DE], errors="coerce")

    return df

def pick_features(df: pd.DataFrame, candidates):
    cols = [c for c in candidates if c in df.columns]
    if len(cols) == 0:
        raise ValueError(",.")
    return cols


# Dataset splitting

Create group-aware train, validation, and test partitions to avoid formulation-level leakage.


In [ ]:
# 2) Group split
def group_split(df: pd.DataFrame,
                group_cols,
                test_size=0.2,
                val_size=0.2,
                random_state=42,
                na_token="<NA>"):
    df = df.copy()
    group_cols = [c for c in group_cols if c in df.columns]
    if len(group_cols) == 0:
        raise ValueError("group_cols No group columns are present in the dataset; group split cannot be performed.")

    tmp = df[group_cols].copy()
    tmp = tmp.replace({np.nan: na_token}).astype(str)

    gid = tmp.apply(
        lambda r: hashlib.md5("||".join(r.values).encode("utf-8")).hexdigest(),
        axis=1
)
    df["_group_id"] = gid
    unique_groups = df["_group_id"].dropna().unique()

    g_trainval, g_test = train_test_split(
        unique_groups, test_size=test_size, random_state=random_state
)
    val_in_trainval = val_size / (1 - test_size)
    g_train, g_val = train_test_split(
        g_trainval, test_size=val_in_trainval, random_state=random_state
)

    df_train = df[df["_group_id"].isin(g_train)].drop(columns=["_group_id"])
    df_val   = df[df["_group_id"].isin(g_val)].drop(columns=["_group_id"])
    df_test  = df[df["_group_id"].isin(g_test)].drop(columns=["_group_id"])

    return df_train, df_val, df_test


# High-delivery label construction

Construct the high-delivery binary label from the training-set delivery quantile.


# Feature engineering

Create optional interaction features and configure one-hot encoding, scaling, and training-set imbalance handling.


In [ ]:
# 3) Interaction features
def add_interactions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Interaction features are constructed only from input features and do not use DE information.

    Use the four predefined interaction features.:
    1) Admin_X_Shape__Plate: Admin * I(Shape == Plate)
    2) Admin_X_TM__Xenograft Orthotopic: Admin * I(TM == Xenograft Orthotopic)
    3) Type_X_CT: Type + "_" + CT   (categorical)
    4) Size_X_MAT__Hydrogel: Size * I(MAT == Hydrogel)
    """
    df = df.copy()

    need_cols = ["Size", "MAT", "CT", "Type", "TM", "Shape", "Admin"]
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        raise KeyError(f"add_interactions Missing columns: {miss}")

    # Ensure clean categorical values and consistent semantic mappings, especially for TM/MAT.
    df = normalize_categorical_df(df, ["Type", "MAT", "TS", "CT", "TM", "Shape", "Admin"])
    df = apply_semantic_mapping(df)

    # numeric
    size  = pd.to_numeric(df["Size"], errors="coerce")
    admin = pd.to_numeric(df["Admin"], errors="coerce")

    # Categorical values after standardization
    mat   = df["MAT"].astype(str)
    ct    = df["CT"].astype(str)
    type_ = df["Type"].astype(str)
    tm    = df["TM"].astype(str)
    shape = df["Shape"].astype(str)

    # 1) Admin_X_Shape__Plate
    df["Admin_X_Shape__Plate"] = np.where(shape == "Plate", admin, 0)

    # 2) Admin_X_TM__Xenograft Orthotopic
    df["Admin_X_TM__Xenograft Orthotopic"] = np.where(tm == "Xenograft Orthotopic", admin, 0)

    # 3) Type_X_CT(categorical interaction)
    df["Type_X_CT"] = type_ + "_" + ct

    # 4) Size_X_MAT__Hydrogel
    df["Size_X_MAT__Hydrogel"] = np.where(mat == "Hydrogel", size, 0)

    # Ensure numeric interaction columns are numeric
    for c in ["Admin_X_Shape__Plate", "Admin_X_TM__Xenograft Orthotopic", "Size_X_MAT__Hydrogel"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


In [ ]:
# 4) One-hot encoding, scaling, and preprocessing
def make_onehot_encoder(min_freq=2):
    try:
        return OneHotEncoder(
            drop="first",
            sparse_output=False,
            handle_unknown="infrequent_if_exist",
            min_frequency=min_freq
)
    except TypeError:
        # sklearnCompatibility with older scikit-learn versions
        return OneHotEncoder(
            drop="first",
            sparse=False,
            handle_unknown="infrequent_if_exist",
            min_frequency=min_freq
)

def build_preprocessor(numeric_features, categorical_features, scaler_mode="standard"):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]

    if scaler_mode == "standard":
        numeric_steps.append(("scaler", StandardScaler()))
    elif scaler_mode == "minmax":
        numeric_steps.append(("scaler", MinMaxScaler()))
    elif scaler_mode == "robust":
        numeric_steps.append(("scaler", RobustScaler()))
    elif scaler_mode == "none":
        pass
    else:
        raise ValueError(f" scaler_mode: {scaler_mode}")

    numeric_transformer = Pipeline(steps=numeric_steps)
    categorical_transformer = make_onehot_encoder(min_freq=OHE_MIN_FREQUENCY)

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
])
    return preprocessor


In [ ]:
# 5) Training-set imbalance handling
def random_oversample(X, y, seed=42):
    rng = np.random.default_rng(seed)
    y = pd.Series(y).reset_index(drop=True)
    X = np.asarray(X)

    vc = y.value_counts()
    if len(vc) < 2:
        return X, y.to_numpy()

    major = vc.idxmax()
    minor = vc.idxmin()
    n_major = vc.max()
    n_minor = vc.min()
    if n_major == n_minor:
        return X, y.to_numpy()

    X_major = X[y == major]
    X_minor = X[y == minor]
    y_major = y[y == major]
    y_minor = y[y == minor]

    add_idx = rng.choice(len(X_minor), size=(n_major - n_minor), replace=True)
    X_minor_up = np.vstack([X_minor, X_minor[add_idx]])
    y_minor_up = np.concatenate([y_minor.to_numpy(), y_minor.iloc[add_idx].to_numpy()])

    X_bal = np.vstack([X_major, X_minor_up])
    y_bal = np.concatenate([y_major.to_numpy(), y_minor_up])

    perm = rng.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

def compute_class_weight_dict(y):
    y = pd.Series(y)
    vc = y.value_counts()
    if len(vc) < 2:
        return None
    total = vc.sum()
    # Simple inverse-frequency weighting
    w = {cls: total/(len(vc)*cnt) for cls, cnt in vc.items()}
    return w


In [ ]:
# 6) Automatic scaler selection
def pick_best_scaler_by_val_auc(X_train_raw, y_train, X_val_raw, y_val,
                                numeric_features, categorical_features,
                                candidates=("standard", "minmax", "robust")):
    best = None
    best_auc = -np.inf
    report = []

    for mode in candidates:
        pre = build_preprocessor(numeric_features, categorical_features, scaler_mode=mode)
        Xtr = pre.fit_transform(X_train_raw)
        Xva = pre.transform(X_val_raw)

        # Use a lightweight logistic regression only to select the scaler; this is not the final model.
        lr = LogisticRegression(max_iter=2000, class_weight="balanced")
        lr.fit(Xtr, y_train)
        p = lr.predict_proba(Xva)[:, 1]

        try:
            auc = roc_auc_score(y_val, p)
        except Exception:
            auc = np.nan
        try:
            ap = average_precision_score(y_val, p)
        except Exception:
            ap = np.nan

        report.append((mode, auc, ap))

        if np.isfinite(auc) and auc > best_auc:
            best_auc = auc
            best = mode

    rep_df = pd.DataFrame(report, columns=["scaler", "val_auc", "val_ap"]).sort_values("val_auc", ascending=False)
    return best, rep_df


In [ ]:
# 7) Main workflow: load, clean, split, label, featurize, preprocess, and handle imbalance.
df_raw = pd.read_csv(PATH_2024, encoding="utf-8-sig")
print("Raw 2024 dataset shape:", df_raw.shape)

df = clean_2024(df_raw)
print("Shape after cleaning:", df.shape)

# Select feature columns
base_features = pick_features(df, FEATURE_CANDIDATES)
print("Base feature columns used:", base_features)

# Group-aware split
df_train, df_val, df_test = group_split(
    df,
    group_cols=GROUP_COLS,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    na_token="<NA>"
)

print("\n2024 group-aware split:")
print("  train:", df_train.shape)
print("  val:", df_val.shape)
print("  test:", df_test.shape)

# The threshold is calculated only from the training set
thr = df_train[TARGET_DE].quantile(QUANTILE_P)
print(f"\nthreshold thr = Q{QUANTILE_P}({TARGET_DE}) on TRAIN = {thr:.6f}")

# Construct labels
for _d in [df_train, df_val, df_test]:
    _d["y"] = (_d[TARGET_DE] >= thr).astype(int)

print("train Class distribution:", df_train["y"].value_counts().to_dict())
print("val   Class distribution:", df_val["y"].value_counts().to_dict())
print("test  Class distribution:", df_test["y"].value_counts().to_dict())

# Construct features
if USE_INTERACTIONS:
    X_train_raw = add_interactions(df_train[base_features].copy())
    X_val_raw   = add_interactions(df_val[base_features].copy())
    X_test_raw  = add_interactions(df_test[base_features].copy())

    # Numeric/categorical feature definitions including the predefined interactions
    numeric_features = [
        "Size", "Zeta Potential", "Admin",
        "Admin_X_Shape__Plate",
        "Admin_X_TM__Xenograft Orthotopic",
        "Size_X_MAT__Hydrogel",
]
    categorical_features = [
        "Type", "MAT", "TS", "CT", "TM", "Shape",
        "Type_X_CT",
]
else:
    X_train_raw = df_train[base_features].copy()
    X_val_raw   = df_val[base_features].copy()
    X_test_raw  = df_test[base_features].copy()

    numeric_features = ["Size", "Zeta Potential", "Admin"]
    categorical_features = ["Type", "MAT", "TS", "CT", "TM", "Shape"]

# Keep only columns that are present
numeric_features = [c for c in numeric_features if c in X_train_raw.columns]
categorical_features = [c for c in categorical_features if c in X_train_raw.columns]

y_train = df_train["y"].to_numpy()
y_val   = df_val["y"].to_numpy()
y_test  = df_test["y"].to_numpy()

print("\n=== Feature column summary ===")
print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

# automatic scaler selection
if SCALER_MODE == "auto":
    best_scaler, rep = pick_best_scaler_by_val_auc(
        X_train_raw, y_train, X_val_raw, y_val,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        candidates=("standard", "minmax", "robust")
)
    print("\n[auto scaler] candidate:")
    print(rep)
    SCALER_MODE_FINAL = best_scaler if best_scaler is not None else "standard"
    print(f"[auto scaler]: {SCALER_MODE_FINAL}")
    rep.to_csv(os.path.join(SAVE_DIR, "scaler_auto_report.csv"), index=False, encoding="utf-8-sig")
else:
    SCALER_MODE_FINAL = SCALER_MODE

# Preprocessing(train fit)
preprocessor = build_preprocessor(numeric_features, categorical_features, scaler_mode=SCALER_MODE_FINAL)
X_train = preprocessor.fit_transform(X_train_raw)
X_val   = preprocessor.transform(X_val_raw)
X_test  = preprocessor.transform(X_test_raw)

print("\nPreprocessing:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

# imbalance handling(training set)
if IMBALANCE_METHOD == "random_oversample":
    X_train_bal, y_train_bal = random_oversample(X_train, y_train, seed=RANDOM_STATE)
    class_weight = None
    print("\n[] random_oversample distribution:", pd.Series(y_train_bal).value_counts().to_dict())
elif IMBALANCE_METHOD == "class_weight":
    X_train_bal, y_train_bal = X_train, y_train
    class_weight = compute_class_weight_dict(y_train)
    print("\n[] class_weight =", class_weight)
elif IMBALANCE_METHOD == "none":
    X_train_bal, y_train_bal = X_train, y_train
    class_weight = None
    print("\n[] ")
else:
    raise ValueError(f" IMBALANCE_METHOD: {IMBALANCE_METHOD}")


In [ ]:
# 8) Strict validation checks
def _assert_no_nan_inf(X, name="X"):
    X_arr = np.asarray(X)
    if np.isnan(X_arr).any():
        raise ValueError(f"{name} contains NaN; check missing-value handling and type conversion.")
    if np.isinf(X_arr).any():
        raise ValueError(f"{name} contains Inf; check outliers or division-by-zero operations.")

def _assert_two_classes(y, name="y"):
    y = np.asarray(y)
    uniq = np.unique(y)
    if len(uniq) < 2:
        raise ValueError(f"{name} has only one class {uniq},binary model training/evaluation is not possible.")

def _row_hash_df(df_: pd.DataFrame, cols, na_token="<NA>") -> pd.Series:
    cols = [c for c in cols if c in df_.columns]
    tmp = df_[cols].copy()
    tmp = tmp.replace({np.nan: na_token}).astype(str)
    return tmp.apply(lambda r: hashlib.md5("||".join(r.values).encode("utf-8")).hexdigest(), axis=1)

print("Starting strict validation checks.")

_assert_two_classes(y_train, "y_train")
_assert_two_classes(y_val, "y_val")
_assert_two_classes(y_test, "y_test")

_assert_no_nan_inf(X_train, "X_train")
_assert_no_nan_inf(X_val, "X_val")
_assert_no_nan_inf(X_test, "X_test")

# Check formulation-level leakage using GROUP_COLS
h_tr = set(_row_hash_df(df_train, GROUP_COLS))
h_va = set(_row_hash_df(df_val, GROUP_COLS))
h_te = set(_row_hash_df(df_test, GROUP_COLS))

print("Formulation overlap check:")
print("  overlap_train_val  =", len(h_tr & h_va))
print("  overlap_train_test =", len(h_tr & h_te))
print("  overlap_val_test   =", len(h_va & h_te))

if (len(h_tr & h_va) > 0) or (len(h_tr & h_te) > 0) or (len(h_va & h_te) > 0):
    raise ValueError("Detected formulation overlap across train/val/test splits; check GROUP_COLS.")

print("Validation passed; proceed to model training.")


In [ ]:
# Key objects for downstream model training and screening
# X_train_bal, y_train_bal   -> Used for training, possibly after imbalance handling
# X_val, y_val               -> Validation
# X_test, y_test             -> Test
# class_weight               ->  IMBALANCE_METHOD="class_weight",

print("final output objects:")
print("X_train_bal:", np.asarray(X_train_bal).shape, "y_train_bal:", len(y_train_bal))
print("X_val:", np.asarray(X_val).shape,       "y_val:", len(y_val))
print("X_test:", np.asarray(X_test).shape,      "y_test:", len(y_test))
print("class_weight:", class_weight)
print("SCALER_MODE_FINAL:", SCALER_MODE_FINAL)
print("USE_INTERACTIONS:", USE_INTERACTIONS)
print("IMBALANCE_METHOD:", IMBALANCE_METHOD)
print("Output directory:", SAVE_DIR)


# Model training

Train and optimize the three paper-comparison models using the finalized settings: LightGBM, XGBoost, and CatBoost.


## LightGBM

LightGBM model optimization and validation/test evaluation.


In [ ]:
# ============================================================
# LightGBM + Optuna(full-metric version:AP,P@K,R@K,TP@K,Lift@K,ROC-AUC,PRF1,Accuracy)
# 1) Optuna optimization objective:5-fold training cross-validation  F1(predict_proba + threshold)
# 2) final reporting:Val/Test report all metrics()
# 3) threshold:fixed=0.5  val_opt_f1(only affects Precision/Recall/F1/Accuracy)
# 4) data variables:strictly use the outputs from data preparation:
#    X_train_bal, y_train_bal, X_val, y_val, X_test, y_test, IMBALANCE_METHOD, class_weight
# 5) "compatible", X_train_use / y_train_use
# ============================================================

import os
import numpy as np
import optuna

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# optional:sparse-matrix detection
try:
    import scipy.sparse as sp
except Exception:
    sp = None


# =========================
# 0)
# =========================
MODEL_FAMILY_NAME = "LightGBM+Optuna"

N_TRIALS = 50
N_SPLITS = 5
RANDOM_STATE = 42

# ---- Top-K()----
K_LIST = [5, 10, 20]   # Outputs
# notes: Optuna(Optuna  F1), K_OPT

# ---- threshold(used for Precision/Recall/F1/Accuracy)----
THRESHOLD_MODE = "fixed"        # "fixed" or "val_opt_f1"
FIXED_PROBA_THRESHOLD = 0.5     # THRESHOLD_MODE="fixed"
VAL_THR_GRID = np.linspace(0.0, 1.0, 101)  # 0.00,0.01,.,1.00

print("\n=========================")
print("training module:", MODEL_FAMILY_NAME)
print("N_TRIALS =", N_TRIALS, "| N_SPLITS =", N_SPLITS, "| RANDOM_STATE =", RANDOM_STATE)
print("THRESHOLD_MODE =", THRESHOLD_MODE, "| FIXED_PROBA_THRESHOLD =", FIXED_PROBA_THRESHOLD)
print("K_LIST =", K_LIST)
print("IMBALANCE_METHOD =", IMBALANCE_METHOD)
print("=========================\n")


# =========================
# 1) data entry(strictly use the data-preparation variables)
# =========================
y_train_bal = np.asarray(y_train_bal).astype(int)
y_val = np.asarray(y_val).astype(int)
y_test = np.asarray(y_test).astype(int)

def _is_sparse(X) -> bool:
    return (sp is not None) and sp.issparse(X)

IS_SPARSE = _is_sparse(X_train_bal)

print("[LGBM data-source confirmation]")
print("X_train_bal:", getattr(X_train_bal, "shape", None), "y_train_bal:", len(y_train_bal))
print("X_val:", getattr(X_val, "shape", None),       "y_val:", len(y_val))
print("X_test:", getattr(X_test, "shape", None),      "y_test:", len(y_test))
print("whether the feature matrix is sparse:", IS_SPARSE)

def row_slice(X, idx):
    """compatible numpy / scipy.sparse / pandas(DataFrame/Series) """
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]

def _assert_two_classes(y, name="y"):
    uniq = np.unique(np.asarray(y))
    if len(uniq) != 2:
        raise ValueError(f"{name} is not a standard binary target(current classes={uniq})")

def _as_array_dense(X):
    if hasattr(X, "to_numpy"):
        return np.asarray(X.to_numpy())
    return np.asarray(X)

def _assert_no_nan_inf(X, name="X"):
    if _is_sparse(X):
        data = X.data
        if np.isnan(data).any():
            raise ValueError(f"{name}(sparse matrix) NaN")
        if np.isinf(data).any():
            raise ValueError(f"{name}(sparse matrix) Inf")
    else:
        arr = _as_array_dense(X)
        if np.isnan(arr).any():
            raise ValueError(f"{name}  NaN")
        if np.isinf(arr).any():
            raise ValueError(f"{name}  Inf")

_assert_two_classes(y_train_bal, "y_train_bal")
_assert_two_classes(y_val, "y_val")
_assert_two_classes(y_test, "y_test")

_assert_no_nan_inf(X_train_bal, "X_train_bal")
_assert_no_nan_inf(X_val, "X_val")
_assert_no_nan_inf(X_test, "X_test")

if X_train_bal.shape[0] != len(y_train_bal):
    raise ValueError("training set X_train_bal / y_train_bal.")
if X_val.shape[0] != len(y_val):
    raise ValueError("validation set X_val / y_val.")
if X_test.shape[0] != len(y_test):
    raise ValueError("test set X_test / y_test.")


# =========================
# 2) class_weight ()
# =========================
CW_FIT = None
if IMBALANCE_METHOD == "class_weight":
    if class_weight is None:
        raise ValueError("IMBALANCE_METHOD='class_weight'  class_weight=None: class_weight.")
    CW_FIT = class_weight
print("class_weight(used forLGBM fit/CV) =", CW_FIT)


# =========================
# 3)  + folds (fold)
# =========================
def safe_ap(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if y_true.sum() == 0:
        return 0.0
    return float(average_precision_score(y_true, y_score))

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return 0.5
    return float(roc_auc_score(y_true, y_score))

def effective_splits(y, requested):
    y = np.asarray(y).astype(int)
    pos = int(y.sum())
    neg = int(len(y) - pos)
    eff = int(min(requested, pos, neg))
    if eff < 2:
        raise ValueError(f"CV:pos={pos}, neg={neg}, requested_splits={requested}.")
    if eff != requested:
        print(f"[] /,CV folds  {requested}  {eff}(pos={pos}, neg={neg}).")
    return eff


# =========================
# 4) Top-K:TP@K / P@K / R@K / Lift@K
# =========================
def ranking_metrics_at_k(y_true, y_score, k: int):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    n = len(y_true)
    k_eff = int(min(max(int(k), 1), n))
    order = np.argsort(-y_score)
    topk = order[:k_eff]

    tp = int(y_true[topk].sum())
    p_at_k = tp / float(k_eff)

    pos_total = int(y_true.sum())
    r_at_k = tp / float(pos_total) if pos_total > 0 else 0.0

    pos_rate = pos_total / float(n) if n > 0 else 0.0
    lift = (p_at_k / pos_rate) if pos_rate > 0 else 0.0

    return {"tp": tp, "p_at_k": float(p_at_k), "r_at_k": float(r_at_k), "lift_at_k": float(lift)}


def eval_all_metrics_from_prob(y_true, y_prob, thr, k_list):
    """
    Outputs:
    AP(PR-AUC), ROC-AUC, TP@K, P@K, R@K, Lift@K, Precision/Recall/F1, Accuracy
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)

    out = {}
    out["AP(PR-AUC)"] = safe_ap(y_true, y_prob)
    out["ROC-AUC"] = safe_auc(y_true, y_prob)

    for k in k_list:
        m = ranking_metrics_at_k(y_true, y_prob, k)
        out[f"TP@{k}"] = m["tp"]
        out[f"P@{k}"] = m["p_at_k"]
        out[f"R@{k}"] = m["r_at_k"]
        out[f"Lift@{k}"] = m["lift_at_k"]

    y_pred = (y_prob >= float(thr)).astype(int)
    out["Accuracy"] = float(accuracy_score(y_true, y_pred))
    out["Precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["Recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["F1"] = float(f1_score(y_true, y_pred, zero_division=0))

    return out


def pretty_print_report(title, metrics, thr, k_list):
    print(f"\n========== {title} ==========")
    print(f"AP(PR-AUC): {metrics['AP(PR-AUC)']:.4f}")
    print(f"ROC-AUC: {metrics['ROC-AUC']:.4f}")
    for k in k_list:
        print(f"TP@{k}: {metrics[f'TP@{k}']:>3d} | "
              f"P@{k}: {metrics[f'P@{k}']:.4f} | "
              f"R@{k}: {metrics[f'R@{k}']:.4f} | "
              f"Lift@{k}: {metrics[f'Lift@{k}']:.4f}")
    print(f"(thr={thr:.3f}) "
          f"Accuracy={metrics['Accuracy']:.4f} | "
          f"Precision={metrics['Precision']:.4f} | "
          f"Recall={metrics['Recall']:.4f} | "
          f"F1={metrics['F1']:.4f}")


# =========================
# 5) (:)
# =========================
def eval_full_report(model, X, y_true, name: str, thr: float, k_list):
    y_true = np.asarray(y_true).astype(int)
    y_prob = model.predict_proba(X)[:, 1]

    metrics = eval_all_metrics_from_prob(y_true, y_prob, thr, k_list)
    pretty_print_report(name, metrics, thr, k_list)
    metrics["y_prob"] = y_prob
    return metrics


# =========================
# 6) CV-F1(Optuna:predict_proba + threshold)
# =========================
def _cv_f1_with_threshold(X, y, params: dict, thr: float, seed=42, n_splits=5):
    y = np.asarray(y).astype(int)
    eff_splits = effective_splits(y, n_splits)
    cv = StratifiedKFold(n_splits=eff_splits, shuffle=True, random_state=seed)

    f1_list = []
    for tr_idx, va_idx in cv.split(np.zeros(len(y)), y):
        X_tr, X_va = row_slice(X, tr_idx), row_slice(X, va_idx)
        y_tr, y_va = y[tr_idx], y[va_idx]

        if len(np.unique(y_tr)) < 2 or len(np.unique(y_va)) < 2:
            f1_list.append(0.0)
            continue

        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)

        y_prob = model.predict_proba(X_va)[:, 1]
        y_pred = (y_prob >= thr).astype(int)
        f1_list.append(f1_score(y_va, y_pred, zero_division=0))

    return float(np.mean(f1_list))


# =========================
# 7) Optuna:training set CV-F1(threshold 0.5,)
# =========================
def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),

        "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.2),
        "max_depth": trial.suggest_int("max_depth", -1, 20),

        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),

        "class_weight": CW_FIT,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1
    }

    return _cv_f1_with_threshold(
        X=X_train_bal, y=y_train_bal,
        params=params,
        thr=FIXED_PROBA_THRESHOLD,
        seed=RANDOM_STATE, n_splits=N_SPLITS
)

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, catch=(ValueError, TypeError, AssertionError))

best_params = study.best_params.copy()
best_params.update({
    "class_weight": CW_FIT,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1
})

print("\n=========================")
print("Optuna best result(training set CV F1, thr=0.5)")
print("best CV F1 =", study.best_value)
for k, v in best_params.items():
    print(f"  {k}: {v}")
print("class_weight() =", CW_FIT)
print("=========================\n")


# =========================
# 8) fit the final model(best_lgbm)
# =========================
best_lgbm = LGBMClassifier(**best_params)
best_lgbm.fit(X_train_bal, y_train_bal)


# =========================
# 9) threshold:fixed  Val  max F1(only affects PRF1/Acc)
# =========================
def pick_threshold_by_val_f1(y_true, y_prob, grid):
    best_thr = 0.5
    best_f1v = -1.0
    y_true = np.asarray(y_true).astype(int)

    for thr in grid:
        y_pred = (y_prob >= thr).astype(int)
        f1v = f1_score(y_true, y_pred, zero_division=0)
        if f1v > best_f1v:
            best_f1v = f1v
            best_thr = float(thr)

    return best_thr, float(best_f1v)

# Val
val_prob = best_lgbm.predict_proba(X_val)[:, 1]

final_thr = FIXED_PROBA_THRESHOLD
thr_tag = "0.5"

if THRESHOLD_MODE == "val_opt_f1":
    final_thr, best_val_f1 = pick_threshold_by_val_f1(y_val, val_prob, VAL_THR_GRID)
    print("\n[threshold selection]  Val select the threshold on the validation set (max F1)")
    print(f"best Val F1 = {best_val_f1:.4f} | Selected thr = {final_thr:.3f}")
    thr_tag = "val_opt_f1"
elif THRESHOLD_MODE == "fixed":
    final_thr = FIXED_PROBA_THRESHOLD
    thr_tag = "0.5"
else:
    raise ValueError("THRESHOLD_MODE  'fixed'  'val_opt_f1'.")


# =========================
# 10)  + final reporting(Val + Test:;Test)
# =========================
MODEL_NAME = f"best_{MODEL_FAMILY_NAME}(IMBALANCE_METHOD={IMBALANCE_METHOD}, thr={thr_tag})"
print("\nfinal model name:", MODEL_NAME)

# VAL full-metric report
val_metrics = eval_all_metrics_from_prob(y_val, val_prob, final_thr, K_LIST)
pretty_print_report("VAL | " + MODEL_NAME, val_metrics, final_thr, K_LIST)

# TEST full-metric report
test_prob = best_lgbm.predict_proba(X_test)[:, 1]
test_metrics = eval_all_metrics_from_prob(y_test, test_prob, final_thr, K_LIST)
pretty_print_report("TEST | " + MODEL_NAME, test_metrics, final_thr, K_LIST)

#  y_pred / y_prob
y_prob_lgbm = test_prob
y_pred_lgbm = (y_prob_lgbm >= final_thr).astype(int)


## XGBoost

XGBoost model optimization and validation/test evaluation.


In [ ]:
# ============================================================
# XGBoost + Optuna(full-metric version:AP,P@K,R@K,TP@K,Lift@K,ROC-AUC,PRF1,Accuracy)
# 1) Optuna optimization objective:5-fold training cross-validation  F1(predict_proba + fixed threshold 0.5)
# 2) final reporting:Val/Test report all metrics()
# 3) threshold:fixed=0.5  val_opt_f1(only affects Precision/Recall/F1/Accuracy)
# 4) data variables:strictly use the outputs from data preparation:
#    X_train_bal, y_train_bal, X_val, y_val, X_test, y_test, IMBALANCE_METHOD, class_weight
# 5) final model variable:best_xgb
# ============================================================

import os
import numpy as np
import optuna

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# optional:sparse-matrix detection
try:
    import scipy.sparse as sp
except Exception:
    sp = None


# =========================
# 0)
# =========================
MODEL_FAMILY_NAME = "XGBoost+Optuna"

N_TRIALS = 30
N_SPLITS = 5
RANDOM_STATE = 42

# ---- Top-K()----
K_LIST = [5, 10, 20]   # Outputs

# ---- threshold(used for Precision/Recall/F1/Accuracy)----
THRESHOLD_MODE = "fixed"        # "fixed" or "val_opt_f1"
FIXED_PROBA_THRESHOLD = 0.5     # THRESHOLD_MODE="fixed"
VAL_THR_GRID = np.linspace(0.0, 1.0, 101)

print("\n=========================")
print("training module:", MODEL_FAMILY_NAME)
print("N_TRIALS =", N_TRIALS, "| N_SPLITS =", N_SPLITS, "| RANDOM_STATE =", RANDOM_STATE)
print("THRESHOLD_MODE =", THRESHOLD_MODE, "| FIXED_PROBA_THRESHOLD =", FIXED_PROBA_THRESHOLD)
print("K_LIST =", K_LIST)
print("IMBALANCE_METHOD =", IMBALANCE_METHOD)
print("=========================\n")


# =========================
# 1) data entry(strictly use the data-preparation variables)
# =========================
y_train_bal = np.asarray(y_train_bal).astype(int)
y_val = np.asarray(y_val).astype(int)
y_test = np.asarray(y_test).astype(int)

def _is_sparse(X) -> bool:
    return (sp is not None) and sp.issparse(X)

IS_SPARSE = _is_sparse(X_train_bal)

print("[XGB data-source confirmation]")
print("X_train_bal:", getattr(X_train_bal, "shape", None), "y_train_bal:", len(y_train_bal))
print("X_val:", getattr(X_val, "shape", None),       "y_val:", len(y_val))
print("X_test:", getattr(X_test, "shape", None),      "y_test:", len(y_test))
print("whether the feature matrix is sparse:", IS_SPARSE)

def row_slice(X, idx):
    """compatible numpy / scipy.sparse / pandas(DataFrame/Series) """
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]

def _assert_two_classes(y, name="y"):
    uniq = np.unique(np.asarray(y))
    if len(uniq) != 2:
        raise ValueError(f"{name} is not a standard binary target(current classes={uniq})")

def _as_array_dense(X):
    if hasattr(X, "to_numpy"):
        return np.asarray(X.to_numpy())
    return np.asarray(X)

def _assert_no_nan_inf(X, name="X"):
    if _is_sparse(X):
        data = X.data
        if np.isnan(data).any():
            raise ValueError(f"{name}(sparse matrix) NaN")
        if np.isinf(data).any():
            raise ValueError(f"{name}(sparse matrix) Inf")
    else:
        arr = _as_array_dense(X)
        if np.isnan(arr).any():
            raise ValueError(f"{name}  NaN")
        if np.isinf(arr).any():
            raise ValueError(f"{name}  Inf")

_assert_two_classes(y_train_bal, "y_train_bal")
_assert_two_classes(y_val, "y_val")
_assert_two_classes(y_test, "y_test")

_assert_no_nan_inf(X_train_bal, "X_train_bal")
_assert_no_nan_inf(X_val, "X_val")
_assert_no_nan_inf(X_test, "X_test")

if X_train_bal.shape[0] != len(y_train_bal):
    raise ValueError("training set X_train_bal / y_train_bal.")
if X_val.shape[0] != len(y_val):
    raise ValueError("validation set X_val / y_val.")
if X_test.shape[0] != len(y_test):
    raise ValueError("test set X_test / y_test.")


# =========================
# 2) imbalance handling:XGB  scale_pos_weight(class_weight)
# =========================
SCALE_POS_WEIGHT = 1.0
if IMBALANCE_METHOD == "class_weight":
    n_pos = int((y_train_bal == 1).sum())
    n_neg = int((y_train_bal == 0).sum())
    if n_pos == 0:
        raise ValueError("y_train_bal =0, scale_pos_weight.")
    SCALE_POS_WEIGHT = float(n_neg / n_pos)

print("scale_pos_weight(used forXGB fit/CV) =", SCALE_POS_WEIGHT)


# =========================
# 3)  + folds (fold)
# =========================
def safe_ap(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if y_true.sum() == 0:
        return 0.0
    return float(average_precision_score(y_true, y_score))

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return 0.5
    return float(roc_auc_score(y_true, y_score))

def effective_splits(y, requested):
    y = np.asarray(y).astype(int)
    pos = int(y.sum())
    neg = int(len(y) - pos)
    eff = int(min(requested, pos, neg))
    if eff < 2:
        raise ValueError(f"CV:pos={pos}, neg={neg}, requested_splits={requested}.")
    if eff != requested:
        print(f"[] /,CV folds  {requested}  {eff}(pos={pos}, neg={neg}).")
    return eff


# =========================
# 4) Top-K:TP@K / P@K / R@K / Lift@K
# =========================
def ranking_metrics_at_k(y_true, y_score, k: int):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    n = len(y_true)
    k_eff = int(min(max(int(k), 1), n))
    order = np.argsort(-y_score)
    topk = order[:k_eff]

    tp = int(y_true[topk].sum())
    p_at_k = tp / float(k_eff)

    pos_total = int(y_true.sum())
    r_at_k = tp / float(pos_total) if pos_total > 0 else 0.0

    pos_rate = pos_total / float(n) if n > 0 else 0.0
    lift = (p_at_k / pos_rate) if pos_rate > 0 else 0.0

    return {"tp": tp, "p_at_k": float(p_at_k), "r_at_k": float(r_at_k), "lift_at_k": float(lift)}


def eval_all_metrics_from_prob(y_true, y_prob, thr, k_list):
    """
    report all metrics:
    AP(PR-AUC), ROC-AUC, TP@K, P@K, R@K, Lift@K, Precision/Recall/F1, Accuracy
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)

    out = {}
    out["AP(PR-AUC)"] = safe_ap(y_true, y_prob)
    out["ROC-AUC"] = safe_auc(y_true, y_prob)

    for k in k_list:
        m = ranking_metrics_at_k(y_true, y_prob, k)
        out[f"TP@{k}"] = m["tp"]
        out[f"P@{k}"] = m["p_at_k"]
        out[f"R@{k}"] = m["r_at_k"]
        out[f"Lift@{k}"] = m["lift_at_k"]

    y_pred = (y_prob >= float(thr)).astype(int)
    out["Accuracy"] = float(accuracy_score(y_true, y_pred))
    out["Precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["Recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["F1"] = float(f1_score(y_true, y_pred, zero_division=0))

    return out


def pretty_print_report(title, metrics, thr, k_list):
    print(f"\n========== {title} ==========")
    print(f"AP(PR-AUC): {metrics['AP(PR-AUC)']:.4f}")
    print(f"ROC-AUC: {metrics['ROC-AUC']:.4f}")
    for k in k_list:
        print(f"TP@{k}: {metrics[f'TP@{k}']:>3d} | "
              f"P@{k}: {metrics[f'P@{k}']:.4f} | "
              f"R@{k}: {metrics[f'R@{k}']:.4f} | "
              f"Lift@{k}: {metrics[f'Lift@{k}']:.4f}")
    print(f"(thr={thr:.3f}) "
          f"Accuracy={metrics['Accuracy']:.4f} | "
          f"Precision={metrics['Precision']:.4f} | "
          f"Recall={metrics['Recall']:.4f} | "
          f"F1={metrics['F1']:.4f}")


# =========================
# 5) CV-F1(Optuna):predict_proba + fixed threshold
# =========================
def _cv_f1_with_threshold(X, y, params: dict, thr: float, seed=42, n_splits=5):
    y = np.asarray(y).astype(int)
    eff_splits = effective_splits(y, n_splits)
    cv = StratifiedKFold(n_splits=eff_splits, shuffle=True, random_state=seed)

    f1_list = []
    for tr_idx, va_idx in cv.split(np.zeros(len(y)), y):
        X_tr, X_va = row_slice(X, tr_idx), row_slice(X, va_idx)
        y_tr, y_va = y[tr_idx], y[va_idx]

        if len(np.unique(y_tr)) < 2 or len(np.unique(y_va)) < 2:
            f1_list.append(0.0)
            continue

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)

        y_prob = model.predict_proba(X_va)[:, 1]
        y_pred = (y_prob >= thr).astype(int)
        f1_list.append(f1_score(y_va, y_pred, zero_division=0))

    return float(np.mean(f1_list))


# =========================
# 6) Optuna:training set CV-F1(threshold 0.5,)
# =========================
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),

        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),

        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),

        "scale_pos_weight": SCALE_POS_WEIGHT,


        "random_state": RANDOM_STATE,
        "eval_metric": "logloss",
        "n_jobs": -1,
        "verbosity": 0,

        #: xgboost (,)
        # "tree_method": "hist",
    }

    return _cv_f1_with_threshold(
        X=X_train_bal, y=y_train_bal,
        params=params,
        thr=FIXED_PROBA_THRESHOLD,
        seed=RANDOM_STATE, n_splits=N_SPLITS
)

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, catch=(ValueError, TypeError, AssertionError))

best_params = study.best_params.copy()
best_params.update({
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "random_state": RANDOM_STATE,
    "eval_metric": "logloss",
    "n_jobs": -1,
    "verbosity": 0,
})

print("\n=========================")
print("Optuna best result(training set CV F1, thr=0.5)")
print("best CV F1 =", study.best_value)
for k, v in best_params.items():
    print(f"  {k}: {v}")
print("scale_pos_weight =", SCALE_POS_WEIGHT)
print("=========================\n")


# =========================
# 7) fit the final model best_xgb(fit  X_train_bal)
# =========================
best_xgb = XGBClassifier(**best_params)
best_xgb.fit(X_train_bal, y_train_bal)


# =========================
# 8) threshold:fixed  Val  max F1(only affects PRF1/Acc)
# =========================
def pick_threshold_by_val_f1(y_true, y_prob, grid):
    best_thr = 0.5
    best_f1v = -1.0
    y_true = np.asarray(y_true).astype(int)

    for thr in grid:
        y_pred = (y_prob >= thr).astype(int)
        f1v = f1_score(y_true, y_pred, zero_division=0)
        if f1v > best_f1v:
            best_f1v = f1v
            best_thr = float(thr)

    return best_thr, float(best_f1v)

val_prob = best_xgb.predict_proba(X_val)[:, 1]

final_thr = FIXED_PROBA_THRESHOLD
thr_tag = "0.5"

if THRESHOLD_MODE == "val_opt_f1":
    final_thr, best_val_f1 = pick_threshold_by_val_f1(y_val, val_prob, VAL_THR_GRID)
    print("\n[threshold selection]  Val select the threshold on the validation set (max F1)")
    print(f"best Val F1 = {best_val_f1:.4f} | Selected thr = {final_thr:.3f}")
    thr_tag = "val_opt_f1"
elif THRESHOLD_MODE == "fixed":
    final_thr = FIXED_PROBA_THRESHOLD
    thr_tag = "0.5"
else:
    raise ValueError("THRESHOLD_MODE  'fixed'  'val_opt_f1'.")


# =========================
# 9)  + final reporting(Val + Test:;Test)
# =========================
MODEL_NAME = f"best_{MODEL_FAMILY_NAME}(IMBALANCE_METHOD={IMBALANCE_METHOD}, thr={thr_tag})"
print("\nfinal model name:", MODEL_NAME)

val_metrics = eval_all_metrics_from_prob(y_val, val_prob, final_thr, K_LIST)
pretty_print_report("VAL | " + MODEL_NAME, val_metrics, final_thr, K_LIST)

test_prob = best_xgb.predict_proba(X_test)[:, 1]
test_metrics = eval_all_metrics_from_prob(y_test, test_prob, final_thr, K_LIST)
pretty_print_report("TEST | " + MODEL_NAME, test_metrics, final_thr, K_LIST)

#  y_pred / y_prob ()
y_prob_xgb = test_prob
y_pred_xgb = (y_prob_xgb >= final_thr).astype(int)


## CatBoost

CatBoost model optimization and validation/test evaluation.


In [ ]:
# ============================================================
# CatBoost + Optuna(full-metric version:AP,P@K,R@K,TP@K,Lift@K,ROC-AUC,PRF1,Accuracy)
# 1) Optuna optimization objective:5-fold training cross-validation  F1(predict_proba + fixed threshold 0.5)
# 2) final reporting:Val/Test report all metrics()
# 3) threshold:fixed=0.5  val_opt_f1(only affects Precision/Recall/F1/Accuracy)
# 4) data variables:strictly use the outputs from data preparation:
#    X_train_bal, y_train_bal, X_val, y_val, X_test, y_test, IMBALANCE_METHOD
# 5) final model variable:best_cat
# 6):save(.cbm)+ savemeta(json) + savepayload(joblib)
# ============================================================

import os
import json
import re
import joblib
import numpy as np
import optuna
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# optional:sparse-matrix detection
try:
    import scipy.sparse as sp
except Exception:
    sp = None


# =========================
# 0)
# =========================
MODEL_FAMILY_NAME = "CatBoost+Optuna"

N_TRIALS = 30
N_SPLITS = 5
RANDOM_STATE = 42

# ---- Top-K()----
K_LIST = [5, 10, 20]  # Outputs

# ---- threshold(used for Precision/Recall/F1/Accuracy)----
THRESHOLD_MODE = "fixed"        # "fixed" or "val_opt_f1"
FIXED_PROBA_THRESHOLD = 0.5
VAL_THR_GRID = np.linspace(0.0, 1.0, 101)

# save: SAVE_DIR()
SAVE_DIR = globals().get("SAVE_DIR", "./output")
SAVE_MODEL = True
MODEL_SUBDIR = "saved_models"   # save: SAVE_DIR/saved_models/

print("\n=========================")
print("training module:", MODEL_FAMILY_NAME)
print("N_TRIALS =", N_TRIALS, "| N_SPLITS =", N_SPLITS, "| RANDOM_STATE =", RANDOM_STATE)
print("THRESHOLD_MODE =", THRESHOLD_MODE, "| FIXED_PROBA_THRESHOLD =", FIXED_PROBA_THRESHOLD)
print("K_LIST =", K_LIST)
print("IMBALANCE_METHOD =", IMBALANCE_METHOD)
print("SAVE_DIR =", SAVE_DIR)
print("=========================\n")


# =========================
# 1) data entry(strictly use the data-preparation variables)
# =========================
y_train_bal = np.asarray(y_train_bal).astype(int)
y_val = np.asarray(y_val).astype(int)
y_test = np.asarray(y_test).astype(int)

def _is_sparse(X) -> bool:
    return (sp is not None) and sp.issparse(X)

IS_SPARSE = _is_sparse(X_train_bal)

print("[CatBoost data-source confirmation]")
print("X_train_bal:", getattr(X_train_bal, "shape", None), "y_train_bal:", len(y_train_bal))
print("X_val:", getattr(X_val, "shape", None),       "y_val:", len(y_val))
print("X_test:", getattr(X_test, "shape", None),      "y_test:", len(y_test))
print("whether the feature matrix is sparse:", IS_SPARSE)

def row_slice(X, idx):
    """compatible numpy / scipy.sparse / pandas(DataFrame/Series) """
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]

def _assert_two_classes(y, name="y"):
    uniq = np.unique(np.asarray(y))
    if len(uniq) != 2:
        raise ValueError(f"{name} is not a standard binary target(current classes={uniq})")

def _as_array_dense(X):
    if hasattr(X, "to_numpy"):
        return np.asarray(X.to_numpy())
    return np.asarray(X)

def _assert_no_nan_inf(X, name="X"):
    # CatBoost  NaN,,
    if _is_sparse(X):
        data = X.data
        if np.isnan(data).any():
            raise ValueError(f"{name}(sparse matrix) NaN")
        if np.isinf(data).any():
            raise ValueError(f"{name}(sparse matrix) Inf")
    else:
        arr = _as_array_dense(X)
        if np.isnan(arr).any():
            raise ValueError(f"{name}  NaN")
        if np.isinf(arr).any():
            raise ValueError(f"{name}  Inf")

_assert_two_classes(y_train_bal, "y_train_bal")
_assert_two_classes(y_val, "y_val")
_assert_two_classes(y_test, "y_test")

_assert_no_nan_inf(X_train_bal, "X_train_bal")
_assert_no_nan_inf(X_val, "X_val")
_assert_no_nan_inf(X_test, "X_test")

if X_train_bal.shape[0] != len(y_train_bal):
    raise ValueError("training set X_train_bal / y_train_bal.")
if X_val.shape[0] != len(y_val):
    raise ValueError("validation set X_val / y_val.")
if X_test.shape[0] != len(y_test):
    raise ValueError("test set X_test / y_test.")


# =========================
# 2)  + folds (fold)
# =========================
def safe_ap(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if y_true.sum() == 0:
        return 0.0
    return float(average_precision_score(y_true, y_score))

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return 0.5
    return float(roc_auc_score(y_true, y_score))

def effective_splits(y, requested):
    y = np.asarray(y).astype(int)
    pos = int(y.sum())
    neg = int(len(y) - pos)
    eff = int(min(requested, pos, neg))
    if eff < 2:
        raise ValueError(f"CV:pos={pos}, neg={neg}, requested_splits={requested}.")
    if eff != requested:
        print(f"[] /,CV folds  {requested}  {eff}(pos={pos}, neg={neg}).")
    return eff


# =========================
# 3) Top-K:TP@K / P@K / R@K / Lift@K
# =========================
def ranking_metrics_at_k(y_true, y_score, k: int):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    n = len(y_true)
    k_eff = int(min(max(int(k), 1), n))
    order = np.argsort(-y_score)
    topk = order[:k_eff]

    tp = int(y_true[topk].sum())
    p_at_k = tp / float(k_eff)

    pos_total = int(y_true.sum())
    r_at_k = tp / float(pos_total) if pos_total > 0 else 0.0

    pos_rate = pos_total / float(n) if n > 0 else 0.0
    lift = (p_at_k / pos_rate) if pos_rate > 0 else 0.0

    return {"tp": tp, "p_at_k": float(p_at_k), "r_at_k": float(r_at_k), "lift_at_k": float(lift)}


def eval_all_metrics_from_prob(y_true, y_prob, thr, k_list):
    """
    report all metrics:
    AP(PR-AUC), ROC-AUC, TP@K, P@K, R@K, Lift@K, Precision/Recall/F1, Accuracy
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)

    out = {}
    out["AP(PR-AUC)"] = safe_ap(y_true, y_prob)
    out["ROC-AUC"] = safe_auc(y_true, y_prob)

    for k in k_list:
        m = ranking_metrics_at_k(y_true, y_prob, k)
        out[f"TP@{k}"] = m["tp"]
        out[f"P@{k}"] = m["p_at_k"]
        out[f"R@{k}"] = m["r_at_k"]
        out[f"Lift@{k}"] = m["lift_at_k"]

    y_pred = (y_prob >= float(thr)).astype(int)
    out["Accuracy"] = float(accuracy_score(y_true, y_pred))
    out["Precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["Recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["F1"] = float(f1_score(y_true, y_pred, zero_division=0))

    return out


def pretty_print_report(title, metrics, thr, k_list):
    print(f"\n========== {title} ==========")
    print(f"AP(PR-AUC): {metrics['AP(PR-AUC)']:.4f}")
    print(f"ROC-AUC: {metrics['ROC-AUC']:.4f}")
    for k in k_list:
        print(f"TP@{k}: {metrics[f'TP@{k}']:>3d} | "
              f"P@{k}: {metrics[f'P@{k}']:.4f} | "
              f"R@{k}: {metrics[f'R@{k}']:.4f} | "
              f"Lift@{k}: {metrics[f'Lift@{k}']:.4f}")
    print(f"(thr={thr:.3f}) "
          f"Accuracy={metrics['Accuracy']:.4f} | "
          f"Precision={metrics['Precision']:.4f} | "
          f"Recall={metrics['Recall']:.4f} | "
          f"F1={metrics['F1']:.4f}")


# =========================
# 4) Optuna:CV-F1(predict_proba + fixed threshold 0.5)
# =========================
def _cv_f1_with_threshold(X, y, params: dict, thr: float, seed=42, n_splits=5):
    y = np.asarray(y).astype(int)
    eff_splits = effective_splits(y, n_splits)
    cv = StratifiedKFold(n_splits=eff_splits, shuffle=True, random_state=seed)

    f1_list = []
    for tr_idx, va_idx in cv.split(np.zeros(len(y)), y):
        X_tr, X_va = row_slice(X, tr_idx), row_slice(X, va_idx)
        y_tr, y_va = y[tr_idx], y[va_idx]

        if len(np.unique(y_tr)) < 2 or len(np.unique(y_va)) < 2:
            f1_list.append(0.0)
            continue

        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr)

        y_prob = model.predict_proba(X_va)[:, 1]
        y_pred = (y_prob >= thr).astype(int)
        f1_list.append(f1_score(y_va, y_pred, zero_division=0))

    return float(np.mean(f1_list))


# =========================
# 5) Optuna:training set CV-F1(threshold 0.5)
# =========================
def objective(trial):
    params = {
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "iterations": trial.suggest_int("iterations", 120, 900),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),


        "loss_function": "Logloss",
        "random_state": RANDOM_STATE,
        "verbose": 0,
        "thread_count": -1,
    }

    return _cv_f1_with_threshold(
        X=X_train_bal, y=y_train_bal,
        params=params,
        thr=FIXED_PROBA_THRESHOLD,
        seed=RANDOM_STATE, n_splits=N_SPLITS
)

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, catch=(ValueError, TypeError, AssertionError))

best_params = study.best_params.copy()
best_params.update({
    "loss_function": "Logloss",
    "random_state": RANDOM_STATE,
    "verbose": 0,
    "thread_count": -1,
})

print("\n=========================")
print("Optuna best result(training set CV F1, thr=0.5)")
print("best CV F1 =", study.best_value)
for k, v in best_params.items():
    print(f"  {k}: {v}")
print("=========================\n")


# =========================
# 6) fit the final model best_cat(fit  X_train_bal)
# =========================
best_cat = CatBoostClassifier(**best_params)
best_cat.fit(X_train_bal, y_train_bal)


# =========================
# 7) threshold:fixed  Val  max F1(only affects PRF1/Acc)
# =========================
def pick_threshold_by_val_f1(y_true, y_prob, grid):
    best_thr = 0.5
    best_f1v = -1.0
    y_true = np.asarray(y_true).astype(int)

    for thr in grid:
        y_pred = (y_prob >= thr).astype(int)
        f1v = f1_score(y_true, y_pred, zero_division=0)
        if f1v > best_f1v:
            best_f1v = f1v
            best_thr = float(thr)

    return best_thr, float(best_f1v)

val_prob = best_cat.predict_proba(X_val)[:, 1]

final_thr = FIXED_PROBA_THRESHOLD
thr_tag = "0.5"

if THRESHOLD_MODE == "val_opt_f1":
    final_thr, best_val_f1 = pick_threshold_by_val_f1(y_val, val_prob, VAL_THR_GRID)
    print("\n[threshold selection]  Val select the threshold on the validation set (max F1)")
    print(f"best Val F1 = {best_val_f1:.4f} | Selected thr = {final_thr:.3f}")
    thr_tag = "val_opt_f1"
elif THRESHOLD_MODE == "fixed":
    final_thr = FIXED_PROBA_THRESHOLD
    thr_tag = "0.5"
else:
    raise ValueError("THRESHOLD_MODE  'fixed'  'val_opt_f1'.")


# =========================
# 8)  + final reporting(Val/Test:;Test)
# =========================
MODEL_NAME = f"best_{MODEL_FAMILY_NAME}(IMBALANCE_METHOD={IMBALANCE_METHOD}, thr={thr_tag})"
print("\nfinal model name:", MODEL_NAME)

val_metrics = eval_all_metrics_from_prob(y_val, val_prob, final_thr, K_LIST)
pretty_print_report("VAL | " + MODEL_NAME, val_metrics, final_thr, K_LIST)

test_prob = best_cat.predict_proba(X_test)[:, 1]
test_metrics = eval_all_metrics_from_prob(y_test, test_prob, final_thr, K_LIST)
pretty_print_report("TEST | " + MODEL_NAME, test_metrics, final_thr, K_LIST)

#  y_pred / y_prob
y_prob_cat = test_prob
y_pred_cat = (y_prob_cat >= final_thr).astype(int)


# =========================
# 9) save + meta + payload()
# =========================
def make_safe_filename(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"[\\/:*?\"<>|]", "_", s)
    s = re.sub(r"\s+", "_", s)
    return s

if SAVE_MODEL:
    model_out_dir = os.path.join(SAVE_DIR, MODEL_SUBDIR)
    os.makedirs(model_out_dir, exist_ok=True)

    safe_name = make_safe_filename(MODEL_NAME)

    # 9.1 save CatBoost ()
    model_path = os.path.join(model_out_dir, f"{safe_name}.cbm")
    best_cat.save_model(model_path)

    # 9.2 save meta(json:threshold/parameters//)
    meta = {
        "model_family": MODEL_FAMILY_NAME,
        "model_name": MODEL_NAME,
        "imbalance_method": IMBALANCE_METHOD,
        "random_state": RANDOM_STATE,
        "n_trials": N_TRIALS,
        "n_splits": N_SPLITS,
        "k_list": K_LIST,
        "threshold_mode": THRESHOLD_MODE,
        "fixed_proba_threshold": float(FIXED_PROBA_THRESHOLD),
        "final_thr": float(final_thr),
        "thr_tag": thr_tag,
        "best_params": best_params,
        "best_cv_f1_thr05": float(study.best_value),
        "val_report": val_metrics,
        "test_report": test_metrics,
        "model_path": model_path,
        "note": "CatBoost.cbm + meta.json + payload.joblib.Preprocessing(to_model_matrix)save encoder/.",
    }
    meta_path = os.path.join(model_out_dir, f"{safe_name}__meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    # 9.3 save payload(joblib: load)
    payload = {
        "model_family": MODEL_FAMILY_NAME,
        "model_name": MODEL_NAME,
        "model_path": model_path,
        "final_thr": float(final_thr),
        "thr_mode": THRESHOLD_MODE,
        "k_list": K_LIST,
        "imbalance_method": IMBALANCE_METHOD,
        "best_params": best_params,
        "best_cv_f1_thr05": float(study.best_value),
        "val_report": val_metrics,
        "test_report": test_metrics,
        # (optional): joblib  CatBoost
        # compatible,,.cbm
        "model_obj": best_cat,
    }
    payload_path = os.path.join(model_out_dir, f"{safe_name}__payload.joblib")
    joblib.dump(payload, payload_path)

    print("\n=========================")
    print("[Saved Model]")
    print(" - CatBoost model:", model_path)
    print(" - meta json:", meta_path)
    print(" - payload joblib:", payload_path)
    print("=========================\n")

    # 9.4(optional)(,)
    print(":")
    print("  (A)  cbm:")
    print("      from catboost import CatBoostClassifier")
    print("      m = CatBoostClassifier(); m.load_model(r'{}')".format(model_path))
    print("  (B)  payload:")
    print("      import joblib; payload = joblib.load(r'{}')".format(payload_path))
    print("      m = payload.get('model_obj', None)  #  cbm ")
    print("      final_thr = payload['final_thr']")


# Model evaluation

Summarize model-level metrics, apply the finalized selection logic, and export evaluation tables and figures.


In [ ]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# optional:sparse-matrix detection(densify)
try:
    import scipy.sparse as sp
except Exception:
    sp = None

# ============================================================
# 0) pandas:Prevent wide tables from being truncated.
# ============================================================
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)

# =========================
# 1) Threshold strategy; fixed threshold is used by default for comparability.
# =========================
THRESHOLD_MODE = "fixed"          # "fixed" or "val_opt_f1"
FIXED_PROBA_THRESHOLD = 0.5
VAL_THR_GRID = np.linspace(0.0, 1.0, 101)

# Top-K (Module)
K_LIST = [5, 10, 20]

print("\n=========================")
print("Module: Model Summary")
print(f"THRESHOLD_MODE = {THRESHOLD_MODE} | FIXED_PROBA_THRESHOLD = {FIXED_PROBA_THRESHOLD}")
print("K_LIST =", K_LIST)
print("=========================\n")

# =========================
# 2) data entry:Use the common output convention from model-training cells.
# =========================
need_vars = ["X_val", "y_val", "X_test"]
missing = [v for v in need_vars if v not in globals()]
if missing:
    raise NameError(f"Missing variables:{missing}\nPlease run the data preparation stage first,generate X_val/y_val/X_test.")

X_val_use = globals()["X_val"]
y_val_use = np.asarray(globals()["y_val"]).astype(int)

X_test_use = globals()["X_test"]
if "y_test_common" in globals() and globals()["y_test_common"] is not None:
    y_test_src = "y_test_common"
    y_test_use = np.asarray(globals()["y_test_common"]).astype(int)
elif "y_test" in globals() and globals()["y_test"] is not None:
    y_test_src = "y_test"
    y_test_use = np.asarray(globals()["y_test"]).astype(int)
else:
    raise NameError(" y_test_common  y_test().")

# =========================
# 3) Basic checks
# =========================
def _is_sparse_matrix(X):
    return (sp is not None) and sp.issparse(X)

def _assert_two_classes(y, name="y"):
    uniq = np.unique(np.asarray(y))
    if len(uniq) != 2:
        raise ValueError(f"{name} is not a standard binary target(current classes={uniq})")

def _assert_no_nan_inf(X, name="X"):
    if _is_sparse_matrix(X):
        data = X.data
        if np.isnan(data).any():
            raise ValueError(f"{name}(sparse matrix) NaN")
        if np.isinf(data).any():
            raise ValueError(f"{name}(sparse matrix) Inf")
        return

    arr = np.asarray(X.to_numpy()) if hasattr(X, "to_numpy") else np.asarray(X)
    if np.isnan(arr).any():
        raise ValueError(f"{name}  NaN")
    if np.isinf(arr).any():
        raise ValueError(f"{name}  Inf")

def _basic_shape_check(X, y, name="split"):
    if getattr(X, "shape", None) is None:
        raise ValueError(f"{name}  X  shape")
    if X.shape[0] != len(y):
        raise ValueError(f"{name}:X.shape[0]={X.shape[0]} != len(y)={len(y)}")

def _y_dist(y):
    u, c = np.unique(np.asarray(y), return_counts=True)
    return dict(zip(u.tolist(), c.tolist()))

_basic_shape_check(X_val_use, y_val_use, "Val")
_basic_shape_check(X_test_use, y_test_use, "Test")

_assert_two_classes(y_val_use, "y_val")
_assert_two_classes(y_test_use, y_test_src)

_assert_no_nan_inf(X_val_use, "X_val")
_assert_no_nan_inf(X_test_use, "X_test")

print("[data-source confirmation]")
if "X_train_bal" in globals() and "y_train_bal" in globals():
    try:
        y_train_bal = np.asarray(globals()["y_train_bal"]).astype(int)
        print("X_train_bal:", getattr(globals()["X_train_bal"], "shape", None),
              "y_train_bal:", len(y_train_bal), "distribution:", _y_dist(y_train_bal))
    except Exception:
        print("X_train_bal: ()  y_train_bal distribution")

print("X_val:", getattr(X_val_use, "shape", None),
      "y_val:", len(y_val_use), "distribution:", _y_dist(y_val_use))
print("X_test:", getattr(X_test_use, "shape", None),
      f"{y_test_src}:", len(y_test_use), "distribution:", _y_dist(y_test_use))
print()

# =========================
# 4) compatible best_* dict: estimator
# =========================
def _unwrap_estimator(obj, max_depth=5):
    """
    compatible best_*  dict /(dict  dict)
    -  obj  stacking dict(meta_model/base_models/base_keys),
    -  key  value  estimator
    """
    if obj is None or max_depth <= 0:
        return None

    # (dict)
    if not isinstance(obj, dict) and (hasattr(obj, "predict_proba") or hasattr(obj, "decision_function") or hasattr(obj, "predict")):
        return obj

    if isinstance(obj, dict):
        # stacking dict
        if ("meta_model" in obj) and ("base_models" in obj) and ("base_keys" in obj):
            return None

        cand_keys = [
            "model", "estimator", "clf", "classifier",
            "best_model", "final_model", "pipe", "pipeline",
            "sk_model"
]
        for k in cand_keys:
            if k in obj:
                got = _unwrap_estimator(obj.get(k), max_depth=max_depth - 1)
                if got is not None:
                    return got

        for v in obj.values():
            got = _unwrap_estimator(v, max_depth=max_depth - 1)
            if got is not None:
                return got

    return None

# =========================
# 5) "/"(Keras  dense float32)
# =========================
def _is_keras_like(model):
    t = str(type(model)).lower()
    return ("tensorflow" in t) or ("keras" in t)

def _to_dense_float32(X):
    if _is_sparse_matrix(X):
        X = X.toarray()
    if hasattr(X, "to_numpy"):
        X = X.to_numpy()
    return np.asarray(X, dtype=np.float32)

def get_pos_score_general(model, X):
    # Keras / TF DNN
    if hasattr(model, "predict") and _is_keras_like(model):
        X_in = _to_dense_float32(X)
        s = np.asarray(model.predict(X_in, verbose=0)).reshape(-1)
        return np.clip(s, 0.0, 1.0)

    # sklearn-like proba
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X)
            if proba is not None:
                proba = np.asarray(proba)
                if proba.ndim == 2 and proba.shape[1] >= 2:
                    return proba[:, 1].ravel()
                if proba.ndim == 1:
                    return proba.ravel()
        except Exception:
            pass

    # decision_function -> minmax
    if hasattr(model, "decision_function"):
        try:
            s = np.asarray(model.decision_function(X)).ravel()
            s_min, s_max = float(np.min(s)), float(np.max(s))
            if s_max > s_min:
                return (s - s_min) / (s_max - s_min)
        except Exception:
            pass

    return None

# =========================
# 6) Stacking(dict) (compatible best_stacking Outputs)
# =========================
def is_stacking_dict(obj):
    return isinstance(obj, dict) and ("meta_model" in obj) and ("base_models" in obj) and ("base_keys" in obj)

def stacking_predict_proba_from_dict(stacking_obj, X):
    base_keys = list(stacking_obj["base_keys"])
    base_models = stacking_obj["base_models"]
    meta = stacking_obj["meta_model"]

    feats = []
    for k in base_keys:
        m = base_models[k]
        s = get_pos_score_general(m, X)
        if s is None:
            raise RuntimeError(f"Stacking base={k} ({type(m)}) /.")
        feats.append(np.asarray(s).ravel())

    Z = np.vstack(feats).T
    if not hasattr(meta, "predict_proba"):
        raise RuntimeError(f"meta_model ({type(meta)})  predict_proba.")
    return np.asarray(meta.predict_proba(Z)[:, 1]).ravel()

# =========================
# 7):Top-K + (Module)
# =========================
def ranking_metrics_at_k(y_true, y_score, k: int):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).ravel()

    n = len(y_true)
    k_eff = int(min(max(int(k), 1), n))
    order = np.argsort(-y_score)
    topk = order[:k_eff]

    tp = int(y_true[topk].sum())
    p_at_k = tp / float(k_eff)

    pos_total = int(y_true.sum())
    r_at_k = tp / float(pos_total) if pos_total > 0 else 0.0

    pos_rate = pos_total / float(n) if n > 0 else 0.0
    lift = (p_at_k / pos_rate) if pos_rate > 0 else 0.0

    return {"tp": tp, "p_at_k": float(p_at_k), "r_at_k": float(r_at_k), "lift_at_k": float(lift)}

def _safe_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if y_score is None or len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return float(roc_auc_score(y_true, y_score))
    except Exception:
        return np.nan

def _safe_ap(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if y_score is None or len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return float(average_precision_score(y_true, y_score))
    except Exception:
        return np.nan

def pick_threshold_by_val_f1(y_true, y_prob, grid):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    best_thr, best_f1v = 0.5, -1.0
    for thr in grid:
        f1v = f1_score(y_true, (y_prob >= float(thr)).astype(int), zero_division=0)
        if f1v > best_f1v:
            best_f1v = float(f1v)
            best_thr = float(thr)
    return best_thr, best_f1v

def eval_metrics_full(y_true, y_prob, thr, k_list):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    y_pred = (y_prob >= float(thr)).astype(int)

    out = {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "ROC-AUC": _safe_auc(y_true, y_prob),
        "AP(PR-AUC)": _safe_ap(y_true, y_prob),
        "Pred_Pos": int(y_pred.sum()),
        "Prob_min": float(np.min(y_prob)),
        "Prob_mean": float(np.mean(y_prob)),
        "Prob_max": float(np.max(y_prob)),
        "N": int(len(y_true)),
    }

    for k in k_list:
        m = ranking_metrics_at_k(y_true, y_prob, k)
        out[f"TP@{k}"] = m["tp"]
        out[f"P@{k}"] = m["p_at_k"]
        out[f"R@{k}"] = m["r_at_k"]
        out[f"Lift@{k}"] = m["lift_at_k"]

    return out, y_pred

def _slug(name):
    return (
        str(name).lower()
.replace("(", "").replace(")", "")
.replace("-", "_").replace(" ", "_")
.replace("+", "plus")
.replace("/", "_")
.replace(".", "")
.replace(",", "")
)

# =========================
# 8) ("best_")
# =========================
MODEL_REGISTRY = [
    ("SVM",                ["best_svm"]),
    ("RandomForest",       ["best_rf"]),
    ("XGBoost",            ["best_xgb"]),
    ("LightGBM",           ["lgbm_best", "best_lgbm"]),
    ("CatBoost",           ["cat_best", "best_cat"]),
    ("KNN",                ["best_knn"]),
    ("LogisticRegression", ["best_LogisticRegression", "best_lr"]),
    ("DNN",                ["best_DNN", "best_dnn"]),
    ("Voting",             ["best_voting"]),
    ("Stacking",           ["best_stacking"]),
]

# =========================
# 9) (Val/Test)+ Outputs + export
# =========================
rows = []
missing_models = []

for model_name, candidates in MODEL_REGISTRY:
    obj = None
    chosen_var = None
    for vn in candidates:
        if vn in globals() and globals()[vn] is not None:
            obj = globals()[vn]
            chosen_var = vn
            break

    if obj is None:
        missing_models.append((model_name, " / ".join(candidates)))
        continue

    # ----  dict (stacking dict)----
    if isinstance(obj, dict) and (not is_stacking_dict(obj)):
        est = _unwrap_estimator(obj, max_depth=5)
        if est is not None:
            obj_for_pred = est
        else:
            obj_for_pred = obj  #  missing
    else:
        obj_for_pred = obj

    # ----  Val/Test  ----
    try:
        if is_stacking_dict(obj):
            val_prob = stacking_predict_proba_from_dict(obj, X_val_use)
            test_prob = stacking_predict_proba_from_dict(obj, X_test_use)
        else:
            val_prob = get_pos_score_general(obj_for_pred, X_val_use)
            test_prob = get_pos_score_general(obj_for_pred, X_test_use)

        if val_prob is None or test_prob is None:
            # fallback: predict
            val_pred_fallback = np.asarray(obj_for_pred.predict(X_val_use)).astype(int).ravel()
            test_pred_fallback = np.asarray(obj_for_pred.predict(X_test_use)).astype(int).ravel()
            val_prob = None
            test_prob = None
        else:
            val_pred_fallback = None
            test_pred_fallback = None

    except Exception as e:
        missing_models.append((model_name, f"{chosen_var}: {repr(e)}"))
        continue

    print("\n=========================")
    print(f": {model_name} | Source: {chosen_var} | type: {type(obj)}")
    print("=========================")

    # ---- threshold ----
    if THRESHOLD_MODE == "fixed":
        thr_use = float(FIXED_PROBA_THRESHOLD)
        thr_tag = "0.5"
    elif THRESHOLD_MODE == "val_opt_f1":
        if val_prob is None:
            thr_use = float(FIXED_PROBA_THRESHOLD)
            thr_tag = "0.5(fallback_no_prob)"
        else:
            thr_use, best_f1v = pick_threshold_by_val_f1(y_val_use, val_prob, VAL_THR_GRID)
            thr_tag = f"val_opt_f1({thr_use:.3f})"
    else:
        raise ValueError("THRESHOLD_MODE  'fixed'  'val_opt_f1'.")

    # ---- Val ----
    if val_prob is not None:
        val_metrics, val_pred = eval_metrics_full(y_val_use, val_prob, thr_use, K_LIST)
    else:
        #: PRF1/Acc, NaN
        val_pred = val_pred_fallback
        val_metrics = {
            "Accuracy": float(accuracy_score(y_val_use, val_pred)),
            "Precision": float(precision_score(y_val_use, val_pred, zero_division=0)),
            "Recall": float(recall_score(y_val_use, val_pred, zero_division=0)),
            "F1": float(f1_score(y_val_use, val_pred, zero_division=0)),
            "ROC-AUC": np.nan,
            "AP(PR-AUC)": np.nan,
            "Pred_Pos": int(np.sum(val_pred)),
            "Prob_min": np.nan, "Prob_mean": np.nan, "Prob_max": np.nan, "N": int(len(y_val_use)),
        }
        for k in K_LIST:
            val_metrics[f"TP@{k}"] = np.nan
            val_metrics[f"P@{k}"] = np.nan
            val_metrics[f"R@{k}"] = np.nan
            val_metrics[f"Lift@{k}"] = np.nan

    # ---- Test ----
    if test_prob is not None:
        test_metrics, test_pred = eval_metrics_full(y_test_use, test_prob, thr_use, K_LIST)
    else:
        test_pred = test_pred_fallback
        test_metrics = {
            "Accuracy": float(accuracy_score(y_test_use, test_pred)),
            "Precision": float(precision_score(y_test_use, test_pred, zero_division=0)),
            "Recall": float(recall_score(y_test_use, test_pred, zero_division=0)),
            "F1": float(f1_score(y_test_use, test_pred, zero_division=0)),
            "ROC-AUC": np.nan,
            "AP(PR-AUC)": np.nan,
            "Pred_Pos": int(np.sum(test_pred)),
            "Prob_min": np.nan, "Prob_mean": np.nan, "Prob_max": np.nan, "N": int(len(y_test_use)),
        }
        for k in K_LIST:
            test_metrics[f"TP@{k}"] = np.nan
            test_metrics[f"P@{k}"] = np.nan
            test_metrics[f"R@{k}"] = np.nan
            test_metrics[f"Lift@{k}"] = np.nan

    # ---- ()----
    row = {
        "Model": model_name,
        "Source": chosen_var,
        "ThresholdMode": THRESHOLD_MODE,
        "ThresholdUsed": thr_use,
        "ThresholdTag": thr_tag,
    }

    # Val
    for k, v in val_metrics.items():
        row[f"Val_{k}"] = v
    # Test
    for k, v in test_metrics.items():
        row[f"Test_{k}"] = v

    rows.append(row)

    # ---- Outputs ----
    suf = _slug(model_name)
    globals()[f"y_pred_{suf}"] = test_pred
    globals()[f"y_prob_{suf}"] = test_prob if test_prob is not None else None

    if model_name.lower().startswith("voting"):
        globals()["y_pred_voting"] = test_pred
        globals()["y_prob_voting"] = test_prob
        globals()["y_test_common"] = y_test_use

    if model_name.lower().startswith("stacking"):
        globals()["y_pred_stacking"] = test_pred
        globals()["y_prob_stacking"] = test_prob
        globals()["y_test_common"] = y_test_use

# =========================
# 10) Outputs + export("composite-metric ranking")
# =========================
result_df = pd.DataFrame(rows)
if result_df.empty:
    raise RuntimeError(":generate best_ model variables.")

#  composite-metric ranking:"TopKscreening"
# notes:
# 1)  P@20:Top20("screeningTop20")
# 2)  Lift@20:()
# 3)  PR-AUC(AP):ranking(ROC)
# 4)  F1/ROC-AUC
sort_cols = ["Test_P@20", "Test_Lift@20", "Test_AP(PR-AUC)", "Test_F1", "Test_ROC-AUC"]
sort_cols = [c for c in sort_cols if c in result_df.columns]  #:

result_df = result_df.sort_values(
    by=sort_cols,
    ascending=[False] * len(sort_cols)
).reset_index(drop=True)

print("\n=========================")
print("(Composite ranking:{})".format(" -> ".join(sort_cols)))
print("=========================")

try:
    from IPython.display import display, HTML
    display(HTML("<div style='overflow-x:auto; width:100%'>" + result_df.to_html(index=False) + "</div>"))
except Exception:
    print(result_df.to_string(index=False))

if missing_models:
    print("\n/():")
    for mn, need in missing_models:
        print(f"  - {mn}: {need}")

# =========================
# 11) export(SAVE_DIR;)
# =========================
BASE_DIR = globals().get("SAVE_DIR", "figures2026.1.9")
OUT_DIR = os.path.join(BASE_DIR, "model_summary")
os.makedirs(OUT_DIR, exist_ok=True)

csv_path  = os.path.join(OUT_DIR, f"model_metrics_summary_{THRESHOLD_MODE}.csv")
xlsx_path = os.path.join(OUT_DIR, f"model_metrics_summary_{THRESHOLD_MODE}.xlsx")
html_path = os.path.join(OUT_DIR, f"model_metrics_summary_{THRESHOLD_MODE}.html")

result_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
result_df.to_excel(xlsx_path, index=False)
result_df.to_html(html_path, index=False)

print(f"\nexport:\n- {csv_path}\n- {xlsx_path}\n- {html_path}")


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# =========================
# 0) y_true
# =========================
if "y_test_common" in globals() and globals()["y_test_common"] is not None:
    y_true = np.asarray(globals()["y_test_common"]).astype(int).ravel()
    y_src = "y_test_common"
elif "y_test" in globals() and globals()["y_test"] is not None:
    y_true = np.asarray(globals()["y_test"]).astype(int).ravel()
    y_src = "y_test"
else:
    raise NameError(" y_test_common  y_test().")

print(f"[y_true] from {y_src} | n={len(y_true)}")

# =========================
# 1)  result_df,""ranking
# =========================
if "result_df" not in globals() or globals()["result_df"] is None:
    raise NameError(" result_df.export,generateranking result_df.")

result_df = globals()["result_df"].copy()

#  composite-metric ranking()
sort_cols = ["Test_P@20", "Test_Lift@20", "Test_AP(PR-AUC)", "Test_F1", "Test_ROC-AUC"]
sort_cols = [c for c in sort_cols if c in result_df.columns]  #:

if len(sort_cols) == 0:
    raise ValueError("result_df Composite ranking(Test_P@20/Test_Lift@20/Test_AP(PR-AUC)/Test_F1/Test_ROC-AUC).")

result_df = result_df.sort_values(by=sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)

best_model_name = str(result_df.loc[0, "Model"])
print("[Best by Composite ranking] =", best_model_name)
print("Composite ranking:", " -> ".join(sort_cols))

# =========================
# 2)  y_pred_<slug> (preferOutputs _slug)
# =========================
def _slug(name):
    return (
        str(name).lower()
.replace("(", "").replace(")", "")
.replace("-", "_").replace(" ", "_")
.replace("+", "plus")
.replace("/", "_")
.replace(".", "")
.replace(",", "")
)

slug = _slug(best_model_name)

# candidate:(y_pred_{_slug(Model)})
pred_candidates = [f"y_pred_{slug}"]

# compatible(generate)
compat_map = {
    "xgboost": ["y_pred_xgb"],
    "lightgbm": ["y_pred_lgbm"],
    "catboost": ["y_pred_cat"],
    "randomforest": ["y_pred_rf"],
    "logisticregression": ["y_pred_lr"],
    "decisiontree": ["y_pred_dt"],
}
pred_candidates += compat_map.get(slug, [])

y_pred_best = None
pred_var_used = None
for vn in pred_candidates:
    if vn in globals() and globals()[vn] is not None:
        y_pred_best = np.asarray(globals()[vn]).astype(int).ravel()
        pred_var_used = vn
        break

if y_pred_best is None:
    raise NameError(
        f"Best model.:{pred_candidates}\n"
        f",generate y_pred_{slug}(compatible)."
)

if len(y_pred_best) != len(y_true):
    raise ValueError(f"{pred_var_used}:pred={len(y_pred_best)} vs y={len(y_true)}")

print(f"[y_pred] use {pred_var_used} | n={len(y_pred_best)}")

# =========================
# 3) Output directory
# =========================
if "SAVE_DIR" not in globals() or globals()["SAVE_DIR"] is None:
    SAVE_DIR = os.path.join("figures2026.1.9", "confusion_matrices")
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# 4) ()
# =========================
cm = confusion_matrix(y_true, y_pred_best)

plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, cmap="Blues", fmt="d",
            xticklabels=[0, 1], yticklabels=[0, 1])
plt.xlabel("Predicted label")
plt.ylabel("True label")

#  Ranking criteria()
plt.title(f"{best_model_name} - Confusion Matrix (Best by {'->'.join(sort_cols)})")

plt.tight_layout()

save_path = os.path.join(SAVE_DIR, f"confusion_matrix_best_by_composite_{best_model_name}.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"save: {save_path}")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# optional:sparse-matrix detection(densify)
try:
    import scipy.sparse as sp
except Exception:
    sp = None

# =========================
#  (used for/, FINAL)
# =========================
FINAL_MODEL_NAME = "CatBoost"
PLOT_ALL_MODELS = True   # True:;False: CatBoost

# =========================
# 0)  X_test / y_test(prefer:*_use > *_real >;yprefery_test_common)
# =========================
def _pick_test_xy():
    # X
    X = None
    X_src = None
    for n in ["X_test_use", "X_test_real", "X_test"]:
        if n in globals() and globals()[n] is not None:
            X = globals()[n]
            X_src = n
            break

    # y
    y = None
    y_src = None
    for n in ["y_test_common", "y_test_use", "y_test_real", "y_test"]:
        if n in globals() and globals()[n] is not None:
            y = np.asarray(globals()[n]).astype(int).ravel()
            y_src = n
            break

    if X is None or y is None:
        raise NameError(
            "test set: X_test_use/X_test_real/X_test  "
            "y_test_common/y_test_use/y_test_real/y_test."
)
    return X, y, X_src, y_src

X_test_use, y_test_use, X_src, y_src = _pick_test_xy()
print(f"[Test set] X from {X_src}, y from {y_src}, X.shape={getattr(X_test_use, 'shape', None)}, y.shape={y_test_use.shape}")

uniq = np.unique(y_test_use)
if len(uniq) < 2:
    raise ValueError(f"y_test has only one class {uniq}, ROC /  AUC.")

# =========================
# 1):/DF -> dense float32(keras)
# =========================
def _is_sparse_matrix(X):
    return (sp is not None) and sp.issparse(X)

def _to_dense_float32(X):
    if _is_sparse_matrix(X):
        X = X.toarray()
    if hasattr(X, "to_numpy"):
        X = X.to_numpy()
    return np.asarray(X, dtype=np.float32)

# =========================
# 2) compatible best_* dict: estimator
# =========================
def _unwrap_estimator(obj, max_depth=5):
    if obj is None or max_depth <= 0:
        return None

    if not isinstance(obj, dict) and (
        hasattr(obj, "predict_proba") or hasattr(obj, "decision_function") or hasattr(obj, "predict")
):
        return obj

    if isinstance(obj, dict):
        if ("meta_model" in obj) and ("base_models" in obj) and ("base_keys" in obj):
            return None

        cand_keys = [
            "model", "estimator", "clf", "classifier",
            "best_model", "final_model", "pipe", "pipeline",
            "sk_model"
]
        for k in cand_keys:
            if k in obj:
                got = _unwrap_estimator(obj.get(k), max_depth=max_depth - 1)
                if got is not None:
                    return got

        for v in obj.values():
            got = _unwrap_estimator(v, max_depth=max_depth - 1)
            if got is not None:
                return got

    return None

# =========================
# 3):"positive-class probability/"
# =========================
def _is_keras_like(model):
    t = str(type(model)).lower()
    return ("tensorflow" in t) or ("keras" in t)

def get_pos_score_general(model, X):
    # Keras / TF
    if hasattr(model, "predict") and _is_keras_like(model):
        X_in = _to_dense_float32(X)
        s = np.asarray(model.predict(X_in, verbose=0)).reshape(-1)
        s = np.clip(s, 0.0, 1.0)
        return s

    # predict_proba
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X)
            if proba is not None:
                proba = np.asarray(proba)
                if proba.ndim == 2 and proba.shape[1] >= 2:
                    return proba[:, 1].ravel()
                if proba.ndim == 1:
                    return proba.ravel()
        except Exception:
            pass

    # decision_function -> minmax
    if hasattr(model, "decision_function"):
        try:
            s = np.asarray(model.decision_function(X)).ravel().astype(float)
            s_min, s_max = float(np.min(s)), float(np.max(s))
            if s_max > s_min:
                return (s - s_min) / (s_max - s_min)
            return np.zeros_like(s, dtype=float)
        except Exception:
            pass

    return None

# =========================
# 4) compatible Stacking(dict)
# =========================
def is_stacking_dict(obj):
    return isinstance(obj, dict) and ("meta_model" in obj) and ("base_models" in obj) and ("base_keys" in obj)

def stacking_predict_proba_from_dict(stacking_obj, X):
    base_keys = list(stacking_obj["base_keys"])
    base_models = stacking_obj["base_models"]
    meta = stacking_obj["meta_model"]

    feats = []
    for k in base_keys:
        m = base_models[k]
        if isinstance(m, dict) and (not is_stacking_dict(m)):
            m2 = _unwrap_estimator(m, max_depth=5)
            if m2 is not None:
                m = m2

        s = get_pos_score_general(m, X)
        if s is None:
            raise RuntimeError(f"Stacking base={k} ({type(m)}) /.")
        feats.append(np.asarray(s).ravel())

    Z = np.vstack(feats).T
    if not hasattr(meta, "predict_proba"):
        raise RuntimeError(f"meta_model ({type(meta)})  predict_proba.")
    return np.asarray(meta.predict_proba(Z)[:, 1]).ravel()

# =========================
# 5) (prefer y_prob_*, best_*)
# =========================
MODEL_SPECS = {
    "RandomForest": {"prob_vars": ["y_prob_randomforest", "y_prob_rf"], "model_vars": ["best_rf"]},
    "KNN":          {"prob_vars": ["y_prob_knn"],                       "model_vars": ["best_knn"]},
    "SVM":          {"prob_vars": ["y_prob_svm"],                       "model_vars": ["best_svm"]},
    "LightGBM":     {"prob_vars": ["y_prob_lgbm", "y_prob_lightgbm"],   "model_vars": ["lgbm_best", "best_lgbm"]},
    "XGBoost":      {"prob_vars": ["y_prob_xgb"],                       "model_vars": ["best_xgb"]},
    "CatBoost":     {"prob_vars": ["y_prob_cat", "y_prob_catboost"],    "model_vars": ["cat_best", "best_cat"]},
    "LogisticRegression": {"prob_vars": ["y_prob_logisticregression", "y_prob_lr"], "model_vars": ["best_LogisticRegression", "best_lr"]},
    "DNN":          {"prob_vars": ["y_prob_dnn"],                       "model_vars": ["best_DNN", "best_dnn"]},
    "Stacking":     {"prob_vars": ["y_prob_stacking"],                  "model_vars": ["best_stacking"]},
    "Voting":       {"prob_vars": ["y_prob_voting"],                    "model_vars": ["best_voting"]},
}

def _pick_first_existing(names):
    for n in names:
        if n in globals() and globals()[n] is not None:
            return globals()[n], n
    return None, None

def _prob_sanity_check(prob):
    prob = np.asarray(prob).ravel().astype(float)
    if np.isnan(prob).any() or np.isinf(prob).any():
        return False
    return True

def _get_prob_for_model(pretty_name, spec):
    # 1)
    prob, prob_name = _pick_first_existing(spec.get("prob_vars", []))
    if prob is not None:
        prob = np.asarray(prob).ravel()
        if not _prob_sanity_check(prob):
            return None, f"prob_bad(nan/inf):{prob_name}"
        return prob, f"prob:{prob_name}"

    # 2)
    model, model_name = _pick_first_existing(spec.get("model_vars", []))
    if model is None:
        return None, "missing(prob+model)"

    if isinstance(model, dict) and (not is_stacking_dict(model)):
        est = _unwrap_estimator(model, max_depth=5)
        if est is not None:
            model = est

    if is_stacking_dict(model):
        try:
            prob = stacking_predict_proba_from_dict(model, X_test_use)
            if not _prob_sanity_check(prob):
                return None, f"stacking_prob_bad:{model_name}"
            return np.asarray(prob).ravel(), f"stacking_dict:{model_name}"
        except Exception as e:
            return None, f"stacking_dict_fail:{model_name}({type(e).__name__})"

    try:
        prob = get_pos_score_general(model, X_test_use)
        if prob is None:
            return None, f"no_score:{model_name}"
        if not _prob_sanity_check(prob):
            return None, f"score_bad(nan/inf):{model_name}"
        return np.asarray(prob).ravel(), f"model:{model_name}"
    except Exception as e:
        return None, f"score_fail:{model_name}({type(e).__name__})"

# =========================
# 6)  AUC + ROC
# =========================
curves = []
skipped = []

for pretty_name, spec in MODEL_SPECS.items():
    if (not PLOT_ALL_MODELS) and (pretty_name != FINAL_MODEL_NAME):
        continue

    prob, source = _get_prob_for_model(pretty_name, spec)
    if prob is None:
        skipped.append((pretty_name, source))
        continue
    if prob.shape[0] != y_test_use.shape[0]:
        skipped.append((pretty_name, f"len_mismatch({source}): prob={prob.shape[0]} vs y={y_test_use.shape[0]}"))
        continue

    try:
        auc = roc_auc_score(y_test_use, prob)
        fpr, tpr, _ = roc_curve(y_test_use, prob)
        curves.append((pretty_name, auc, fpr, tpr, source))
    except Exception as e:
        skipped.append((pretty_name, f"roc_fail({source}): {type(e).__name__}"))
        continue

if not curves:
    raise RuntimeError(" ROC.")

#  (AUC)
curves.sort(key=lambda x: (x[0] != FINAL_MODEL_NAME, -x[1]))

print("\n[ROC] Used models:")
for name, auc, _, _, src in curves:
    print(f"  - {name:20s} AUC={auc:.6f} ({src})")

if skipped:
    print("\n[ROC] Skipped models:")
    for name, reason in skipped:
        print(f"  - {name}: {reason}")

# =========================
# 7) (figsize, FINAL)
#    -  CatBoost, label  FINAL
# =========================
plt.figure(figsize=(9, 7))
for name, auc, fpr, tpr, src in curves:
    lw = 3 if name == FINAL_MODEL_NAME else 2
    plt.plot(fpr, tpr, lw=lw, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Sorted by AUC)")
plt.grid(True, alpha=0.35)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()

# =========================
# 8) save: confusion_matrix  SAVE_DIR, figures2026.1.9
# =========================
_save_dir = globals().get("SAVE_DIR", None)
if _save_dir is None:
    BASE_DIR = "figures2026.1.9"
else:
    tail = os.path.basename(str(_save_dir)).lower()
    if tail in {"confusion_matrix", "confusion_matrices", "model_summary", "shap", "roc"}:
        BASE_DIR = os.path.dirname(str(_save_dir)) or "figures2026.1.9"
    else:
        BASE_DIR = str(_save_dir)

OUT_DIR = os.path.join(BASE_DIR, "roc")
os.makedirs(OUT_DIR, exist_ok=True)

save_path = os.path.join(OUT_DIR, "models_roc_curve_sorted_auc.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nROC figure saved to: {save_path}")


# SHAP interpretation

Compute and export SHAP-based model interpretation for the selected final model.


In [ ]:
# =================== SHAP analysis(publication-style mean(|SHAP|) + standard beeswarm(=feature value magnitude)) ===================
#  Notes:
# 1) "Best model" PR-AUC,【】composite-metric ranking:
#    Test_P@20 -> Test_Lift@20 -> Test_AP(PR-AUC) -> Test_F1 -> Test_ROC-AUC
# 2) Do not add a FINAL label(FINAL,)
# 3) Keep the original figure sizes:bar figsize=(8.2,6.2);beeswarm plot_size=(8.6,6.4);dpi
# ==================================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import shap

# optional:CatBoost Pool()
try:
    from catboost import Pool
    _HAS_CATBOOST = True
except Exception:
    Pool = None
    _HAS_CATBOOST = False

# =========================
#  0) Set publication fonts:Robust publication font setup(prefer font name -> then try font files)
# =========================
import matplotlib as mpl
from matplotlib import font_manager

def set_chinese_font():
    candidates = [
        "Microsoft YaHei",      # Windows
        "SimHei",               # Windows
        "SimSun",               # Windows
        "Microsoft JhengHei",   # Windows
        "PingFang SC",          # macOS
        "Heiti SC",             # macOS
        "Noto Sans CJK SC",     # Linux
        "WenQuanYi Micro Hei",  # Linux
        "Arial Unicode MS"
]

    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            mpl.rcParams["font.family"] = "sans-serif"
            mpl.rcParams["font.sans-serif"] = [name]
            mpl.rcParams["axes.unicode_minus"] = False
            print(f"[Font] Using Chinese font by name: {name}")
            return name

    # If font names are unavailable, skip file-based registration in the public version.
    font_paths = [
        # Windows
        "",
        "",
        "",
        "",
        # macOS
        "/System/Library/Fonts/PingFang.ttc",
        "/System/Library/Fonts/STHeiti Medium.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        # Linux
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/truetype/arphic/uming.ttc",
]

    for fp in font_paths:
        if os.path.exists(fp):
            try:
                font_manager.fontManager.addfont(fp)
                prop = font_manager.FontProperties(fname=fp)
                name = prop.get_name()
                mpl.rcParams["font.family"] = "sans-serif"
                mpl.rcParams["font.sans-serif"] = [name]
                mpl.rcParams["axes.unicode_minus"] = False
                print(f"[Font] Using Chinese font by file: {fp} -> name: {name}")
                return name
            except Exception as e:
                print(f"[Font] addfont failed: {fp} ({repr(e)})")

    mpl.rcParams["axes.unicode_minus"] = False
    print("[Font] No preferred publication font was found; using matplotlib defaults.")
    return None

set_chinese_font()

# -----------------------

# -----------------------
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 600

# =========================
# ()
# =========================
if "USE_INTERACTIONS" not in globals():
    USE_INTERACTIONS = False
print(f"[Config] USE_INTERACTIONS = {USE_INTERACTIONS}")

# =========================
# 0)  "Best model"(composite-metric ranking,)
# =========================
if "result_df" not in globals() or globals()["result_df"] is None:
    raise NameError(" result_df:generate result_df.")

result_df_use = globals()["result_df"].copy()

need_cols = ["Model", "Source"]
for c in need_cols:
    if c not in result_df_use.columns:
        raise ValueError(f"result_df  {need_cols}.:{list(result_df_use.columns)}")

#  composite-metric ranking()
sort_cols_pref = ["Test_P@20", "Test_Lift@20", "Test_AP(PR-AUC)", "Test_F1", "Test_ROC-AUC"]
sort_cols = [c for c in sort_cols_pref if c in result_df_use.columns]
if len(sort_cols) == 0:
    raise ValueError(
        "result_df Composite ranking."
        f":{sort_cols_pref};:{list(result_df_use.columns)}"
)

best_row = result_df_use.sort_values(by=sort_cols, ascending=[False] * len(sort_cols)).iloc[0]
used_sort = " -> ".join(sort_cols)

best_model_name = str(best_row["Model"])
best_model_src  = str(best_row["Source"])
print(f"Run SHAP interpretation with the best model:{best_model_name}(source variable:{best_model_src},selection criteria:{used_sort})")

# =========================
# 1)  X(model input)
#    prefer:X_test_raw -> preprocessor.transform()
#: X_test(transform)
# =========================
if "preprocessor" not in globals() or globals()["preprocessor"] is None:
    raise NameError(" preprocessor(ColumnTransformer).")

preprocessor = globals()["preprocessor"]

if "X_test_raw" in globals() and globals()["X_test_raw"] is not None:
    X_all = preprocessor.transform(globals()["X_test_raw"])
    X_src = "X_test_raw -> preprocessor.transform"
elif "X_test" in globals() and globals()["X_test"] is not None:
    X_all = globals()["X_test"]
    X_src = "X_test (assumed transformed)"
else:
    raise NameError(" X_test_raw  X_test().")

print(f"[X] from {X_src} | X_all.shape={getattr(X_all,'shape',None)}")

#  -> dense(beeswarm)
X_all_dense = X_all.toarray() if hasattr(X_all, "toarray") else np.asarray(X_all)

# =========================
# 2):prefer get_feature_names_out,""
# =========================
try:
    feature_names_all = list(preprocessor.get_feature_names_out())
except Exception:
    feature_names_all = [f"f{i}" for i in range(X_all_dense.shape[1])]
    print("[] preprocessor.get_feature_names_out(),.")

if len(feature_names_all) != X_all_dense.shape[1]:
    print("[],.")
    feature_names_all = [f"f{i}" for i in range(X_all_dense.shape[1])]

def _beautify_name(n: str) -> str:
    if not isinstance(n, str):
        return str(n)
    if n.startswith("num__"):
        return n.split("num__", 1)[1]
    if n.startswith("cat__"):
        return n.split("cat__", 1)[1]
    return n

feature_names_all = [_beautify_name(n) for n in feature_names_all]

# =========================
# 3): best_*
# =========================
MODEL_VAR_CANDIDATES = {
    "DecisionTree":       ["best_dt"],
    "RandomForest":       ["best_rf"],
    "KNN":                ["best_knn"],
    "SVM":                ["best_svm"],
    "LightGBM":           ["lgbm_best", "best_lgbm"],
    "XGBoost":            ["best_xgb"],
    "CatBoost":           ["cat_best", "best_cat"],
    "LogisticRegression": ["best_LogisticRegression", "best_lr"],
    "DNN":                ["best_DNN", "best_dnn"],
    "Stacking":           ["best_stacking"],   #  dict
    "Voting":             ["best_voting"],
}

def _pick_first_existing(names):
    for n in names:
        if n in globals() and globals()[n] is not None:
            return globals()[n], n
    return None, None

if best_model_name not in MODEL_VAR_CANDIDATES:
    raise KeyError(f"best_model_name={best_model_name}:{list(MODEL_VAR_CANDIDATES.keys())}")

best_model, best_model_var = _pick_first_existing(MODEL_VAR_CANDIDATES[best_model_name])
if best_model is None:
    raise NameError(f"model variables:{MODEL_VAR_CANDIDATES[best_model_name]}")

print(f"[Best model] {best_model_name} <- {best_model_var} | type={type(best_model)}")

# =========================
# 4) "probability output"(/ SHAP)
# =========================
def _is_keras_like(m):
    t = str(type(m)).lower()
    return ("tensorflow" in t) or ("keras" in t)

def predict_pos_proba(m, X):
    if hasattr(m, "predict_proba"):
        p = np.asarray(m.predict_proba(X))
        if p.ndim == 2 and p.shape[1] >= 2:
            return p[:, 1].ravel()
        if p.ndim == 1:
            return p.ravel()

    if hasattr(m, "predict") and _is_keras_like(m):
        s = np.asarray(m.predict(X, verbose=0)).reshape(-1)
        return np.clip(s, 0.0, 1.0)

    if hasattr(m, "decision_function"):
        s = np.asarray(m.decision_function(X)).ravel()
        s_min, s_max = float(np.min(s)), float(np.max(s))
        if s_max > s_min:
            return (s - s_min) / (s_max - s_min)
        return np.zeros_like(s)

    raise RuntimeError(" predict_proba / keras predict / decision_function.")

def is_stacking_dict(obj):
    return isinstance(obj, dict) and ("meta_model" in obj) and ("base_models" in obj) and ("base_keys" in obj)

def stacking_predict_pos_proba_from_dict(stacking_obj, X):
    base_keys = list(stacking_obj["base_keys"])
    base_models = stacking_obj["base_models"]
    meta = stacking_obj["meta_model"]

    feats = []
    for k in base_keys:
        feats.append(predict_pos_proba(base_models[k], X))
    Z = np.vstack(feats).T

    if not hasattr(meta, "predict_proba"):
        raise RuntimeError("Stacking meta_model  predict_proba.")
    return np.asarray(meta.predict_proba(Z)[:, 1]).ravel()

# =========================
# 5) /()
# =========================
TOPK = 20
BG_N = 200
EXPLAIN_N = min(2000, X_all_dense.shape[0])

rng = np.random.RandomState(42)

if X_all_dense.shape[0] > EXPLAIN_N:
    explain_idx = rng.choice(X_all_dense.shape[0], size=EXPLAIN_N, replace=False)
    X_explain = X_all_dense[explain_idx]
else:
    X_explain = X_all_dense

n_bg = min(BG_N, X_all_dense.shape[0])
bg_idx = rng.choice(X_all_dense.shape[0], size=n_bg, replace=False)
X_bg = X_all_dense[bg_idx]

# =========================
# 6)  SHAP: TreeExplainer;/ shap.Explainer(probability output, background)
# =========================
TREE_MODEL_NAMES = {"DecisionTree", "RandomForest", "XGBoost", "LightGBM", "CatBoost"}

print(f"[SHAP] start | model={best_model_name} | X_explain={X_explain.shape} | X_bg={X_bg.shape}")

if best_model_name in TREE_MODEL_NAMES and (not is_stacking_dict(best_model)):
    if best_model_name == "CatBoost" and _HAS_CATBOOST:
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(Pool(X_explain))
    else:
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_explain)


    if isinstance(shap_values, list) and len(shap_values) == 2:
        sv = np.asarray(shap_values[1])
    else:
        sv = np.asarray(shap_values)

    #  base_value
    if sv.shape[1] == X_explain.shape[1] + 1:
        sv = sv[:,:-1]
else:
    #  / Voting / Stacking(dict):"Outputs"
    if best_model_name == "Stacking" and is_stacking_dict(best_model):
        f = lambda X: stacking_predict_pos_proba_from_dict(best_model, X)
    else:
        f = lambda X: predict_pos_proba(best_model, X)

    explainer = shap.Explainer(f, X_bg)
    exp = explainer(X_explain)
    sv = np.asarray(exp.values)
    if sv.ndim == 3:
        sv = sv[:,:, -1]

if sv.shape[1] != X_explain.shape[1]:
    raise RuntimeError(f"SHAP:sv={sv.shape}, X_explain={X_explain.shape}")

print("[OK] sv.shape =", sv.shape)

# =========================
# 7)  USE_INTERACTIONS:interaction features
# =========================
cross_num_feats = ["Size_x_Zeta", "Size_x_TS_Passive", "Zeta_x_Shape_Spherical"]
cross_categ_feats = ["TM_CT", "Shape_Type", "MAT_TS", "Admin_TM", "Admin_CT"]

def _is_interaction_feature(name: str) -> bool:
    if not isinstance(name, str):
        name = str(name)
    #  _x_ ()
    if "_x_" in name:
        return True
    for k in (cross_num_feats + cross_categ_feats):
        if k in name:
            return True
    return False

if USE_INTERACTIONS:
    vis_mask = np.ones(len(feature_names_all), dtype=bool)
else:
    vis_mask = np.array([not _is_interaction_feature(n) for n in feature_names_all], dtype=bool)

X_vis = X_explain[:, vis_mask]
sv_vis = sv[:, vis_mask]
names_vis = [n for n, m in zip(feature_names_all, vis_mask) if m]

print(f"[Visual] features kept = {len(names_vis)} / {len(feature_names_all)}")

# =========================
# 8) publication-style:mean(|SHAP|), TopK
# =========================
global_imp = np.mean(np.abs(sv_vis), axis=0)
order = np.argsort(global_imp)[:-1]
top_idx = order[:min(TOPK, len(names_vis))]

#  TopK(,)
sv_top = sv_vis[:, top_idx]
X_top = X_vis[:, top_idx]
names_top = [names_vis[i] for i in top_idx]
imp_top = global_imp[top_idx]

# Output directory
BASE_DIR = "figures2026.1.9"
OUT_DIR = os.path.join(BASE_DIR, "shap")
os.makedirs(OUT_DIR, exist_ok=True)

tag = "with_interactions" if USE_INTERACTIONS else "no_interactions"

# =========================
# 9) Bar:mean(|SHAP|) TopK()
# =========================
plt.figure(figsize=(8.2, 6.2))
plt.barh(names_top[:-1], imp_top[:-1])
plt.xlabel("mean(|SHAP value|)")
plt.ylabel("Feature")
plt.title(f"{best_model_name} - Global Feature Importance (Top{len(names_top)})")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()

bar_path = os.path.join(OUT_DIR, f"SHAP_mean_abs_bar_top{len(names_top)}_{best_model_name}_{tag}.png")
plt.savefig(bar_path, bbox_inches="tight")
plt.show()
print(f"[Saved] {bar_path}")

# =========================
# 10)   Beeswarm:=feature value magnitude(:features=X_top)
#     compatible SHAP:parameters shap_values
# =========================
plt.figure()
shap.summary_plot(
    sv_top,                       # parameters,compatible
    features=X_top,
    feature_names=names_top,
    max_display=len(names_top),
    show=False,
    plot_size=(8.6, 6.4)
)

ax = plt.gca()
ax.set_title("SHAP Summary (Beeswarm)", pad=10)
ax.grid(axis="x", linestyle="--", alpha=0.18)
ax.axvline(0, color="#444", lw=1.0, alpha=0.85)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.set_xlabel("SHAP value (impact on model output)")
plt.tight_layout()

summary_path = os.path.join(OUT_DIR, f"SHAP_beeswarm_top{len(names_top)}_{best_model_name}_{tag}.png")
plt.savefig(summary_path, bbox_inches="tight")
plt.show()
print(f"[Saved] {summary_path}")

# =========================
# 11) optional:Dependencies(Top1)
# =========================
try:
    shap.dependence_plot(names_top[0], sv_top, X_top, feature_names=names_top, show=True)
except Exception as e:
    print(f"[Info] dependence_plot:{type(e).__name__}: {e}")


# Virtual formulation screening

Estimate data-driven working ranges and define the search space for virtual formulation screening.


In [ ]:
# =================== Target-oriented range estimation and rule export for formulation generation ===================
# Outputs:
# 1) Training threshold thr consistent y labels; use df_train['y'] if available, otherwise construct labels from thr.
# 2) Continuous features:high-delivery(y=1) candidate interval(q10-q90 / q05-q95)+ distribution plot(hue=y)
# 3) Two-dimensional continuous-feature:Size-Zeta / Size-Admin / Zeta-Admin scatter(hue=y)+ high-delivery region box(by quantiles)
# 4) Categorical features:pos_rate + lift (CSV)+ TopLift bar plot
# 5) export search_space.json(used directly for downstream sampling)

import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# seaborn optional:used for(, USE_SEABORN=False)
USE_SEABORN = True
try:
    import seaborn as sns
except Exception:
    USE_SEABORN = False
    sns = None

import matplotlib as mpl
from matplotlib import font_manager


# =========================
# 0) Basic configuration
# =========================
TARGET_DE = globals().get("TARGET_DE", "DE_tumor")

# SAVE_DIR:Use the output directory from the training stage
SAVE_DIR = globals().get("SAVE_DIR", "./output")
os.makedirs(SAVE_DIR, exist_ok=True)

OUT_DIR = os.path.join(SAVE_DIR, "EDA_targeted_for_generation")
os.makedirs(OUT_DIR, exist_ok=True)

# Dataset used for range estimation; train is recommended.
PLOT_ON = "train"   # "train" / "val" / "test" / "all"

# Quantile ranges for high-delivery candidate intervals; robust to outliers.
Q_LOW_MAIN  = 0.10
Q_HIGH_MAIN = 0.90
Q_LOW_WIDE  = 0.05
Q_HIGH_WIDE = 0.95

# Minimum category count threshold:Minimum count required for pos_rate/lift ranking
CAT_MIN_COUNT = 5

# "recommended category set"threshold(lift>1 indicates better than the overall positive rate)
REC_LIFT_MIN = 1.20

# (Project-specific)
NUM_COLS = ["Size", "Zeta Potential", "Admin"]
CAT_COLS = ["Type", "MAT", "TS", "CT", "TM", "Shape"]

SHOW_PLOTS = True  # True=save+;False=save


# =========================
# 1) Robust publication font setup
# =========================
def set_chinese_font():
    candidates = [
        "Microsoft YaHei", "SimHei", "SimSun", "Microsoft JhengHei",
        "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "WenQuanYi Micro Hei",
        "Arial Unicode MS"
]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            mpl.rcParams["font.family"] = "sans-serif"
            mpl.rcParams["font.sans-serif"] = [name]
            mpl.rcParams["axes.unicode_minus"] = False
            return name


    font_paths = [
        "",
        "",
        "",
        "/System/Library/Fonts/PingFang.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
]
    for fp in font_paths:
        if os.path.exists(fp):
            try:
                font_manager.fontManager.addfont(fp)
                prop = font_manager.FontProperties(fname=fp)
                name = prop.get_name()
                mpl.rcParams["font.family"] = "sans-serif"
                mpl.rcParams["font.sans-serif"] = [name]
                mpl.rcParams["axes.unicode_minus"] = False
                return name
            except Exception:
                pass

    mpl.rcParams["axes.unicode_minus"] = False
    return None

set_chinese_font()
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 600


# =========================
# 2):prefer df_train/df_val/df_test, df
# =========================
def _pick_df(name: str):
    if name in globals() and globals()[name] is not None:
        return globals()[name].copy()
    return None

df_train = _pick_df("df_train")
df_val   = _pick_df("df_val")
df_test  = _pick_df("df_test")
df_all   = _pick_df("df")

#  df_all  train/val/test, all(used for,used for"")
if df_all is None and any(d is not None for d in [df_train, df_val, df_test]):
    parts = [d for d in [df_train, df_val, df_test] if d is not None]
    df_all = pd.concat(parts, axis=0, ignore_index=True)

if df_all is None and df_train is None and df_val is None and df_test is None:
    raise NameError(" DataFrame: df_train/df_val/df_test/df.")

def _choose_plot_df(which: str):
    which = str(which).lower().strip()
    if which == "train" and df_train is not None:
        return df_train, "df_train"
    if which == "val" and df_val is not None:
        return df_val, "df_val"
    if which == "test" and df_test is not None:
        return df_test, "df_test"
    if which == "all" and df_all is not None:
        return df_all, "df_all"

    #:train -> val -> all -> test
    for d, name in [(df_train, "df_train"), (df_val, "df_val"), (df_all, "df_all"), (df_test, "df_test")]:
        if d is not None:
            return d, name
    raise RuntimeError(": df.")

df_plot, df_plot_name = _choose_plot_df(PLOT_ON)
print(f"[PLOT_ON] {PLOT_ON} -> {df_plot_name} | shape={df_plot.shape}")


# =========================
# 3):,
# =========================
def _normalize_cat_text(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    s = s.str.replace("\u00a0", " ", regex=False)
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.strip()
    return s

def normalize_categorical_df(df: pd.DataFrame, cat_cols) -> pd.DataFrame:
    df = df.copy()
    for c in cat_cols:
        if c in df.columns:
            df[c] = _normalize_cat_text(df[c])
            df[c] = df[c].replace({"Others": "Other", "others": "Other"})
    # cervix (optional)
    if "CT" in df.columns:
        df["CT"] = df["CT"].astype(str).str.replace(r"^\s*cervix\s*$", "Cervix", regex=True, flags=re.IGNORECASE)
    return df

df_plot = normalize_categorical_df(df_plot, CAT_COLS)


# =========================
# 4) threshold thr:prefer thr(thr_real)
# =========================
if "thr_real" in globals() and globals()["thr_real"] is not None:
    thr = float(globals()["thr_real"])
    thr_src = "thr_real()"
elif "thr" in globals() and globals()["thr"] is not None:
    thr = float(globals()["thr"])
    thr_src = "thr()"
else:
    #: df_train  0.75 (,)
    if df_train is not None and TARGET_DE in df_train.columns:
        thr = float(pd.to_numeric(df_train[TARGET_DE], errors="coerce").quantile(0.75))
        thr_src = "fallback: df_train Q0.75"
    else:
        thr = float(pd.to_numeric(df_plot[TARGET_DE], errors="coerce").quantile(0.75))
        thr_src = "fallback: df_plot Q0.75"

print(f"[thr] {thr:.6f} |:{thr_src}")


# =========================
# 5) y:prefer df  y; thr generate y_thr
# =========================
def ensure_binary_label(df: pd.DataFrame, target_de: str, thr_value: float, prefer_col="y"):
    df = df.copy()
    if prefer_col in df.columns:
        y = pd.to_numeric(df[prefer_col], errors="coerce")
        #  0/1
        uniq = set(y.dropna().unique().tolist())
        if uniq.issubset({0, 1}):
            df["_y_used_"] = y.astype(int)
            df["_y_src_"] = prefer_col
            return df

    # threshold
    de = pd.to_numeric(df[target_de], errors="coerce")
    df["_y_used_"] = (de >= float(thr_value)).astype(int)
    df["_y_src_"] = f"{target_de}>={thr_value:.6f}"
    return df

if TARGET_DE not in df_plot.columns:
    raise NameError(f"{df_plot_name}  {TARGET_DE},:{list(df_plot.columns)[:20]}.")

df_plot = ensure_binary_label(df_plot, TARGET_DE, thr, prefer_col="y")
y_col = "_y_used_"
print(f"[y]:{df_plot['_y_src_'].iloc[0]} | pos_rate={df_plot[y_col].mean():.3f} | n={len(df_plot)}")


# =========================
# 6) Continuous features:candidate interval + distribution plot(hue=y)
# =========================
num_cols_use = [c for c in NUM_COLS if c in df_plot.columns]
if len(num_cols_use) == 0:
    raise NameError(f"{df_plot_name} Numeric columns(candidate:{NUM_COLS}).")

def _to_num(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

#  Size  log10(nm),generate Size_nm (, Size)
if "Size" in df_plot.columns:
    s = _to_num(df_plot["Size"])
    if s.notna().mean() > 0.6 and (s.dropna().between(-1, 6).mean() > 0.85):
        df_plot["Size_nm"] = (10.0 ** s).round(3)
        print("[Info]  Size  log10(nm),generate Size_nm used for.")

# generate
numeric_rules = []
for col in num_cols_use + (["Size_nm"] if "Size_nm" in df_plot.columns else []):
    x = _to_num(df_plot[col])
    y = df_plot[y_col].astype(int)
    x_pos = x[y == 1].dropna()
    x_all = x.dropna()

    if len(x_pos) < 5 or len(x_all) < 10:
        continue

    row = {
        "feature": col,
        "n_all": int(x_all.shape[0]),
        "n_pos": int(x_pos.shape[0]),
        "pos_rate": float(y.mean()),
        "pos_q05": float(x_pos.quantile(Q_LOW_WIDE)),
        "pos_q10": float(x_pos.quantile(Q_LOW_MAIN)),
        "pos_med": float(x_pos.median()),
        "pos_q90": float(x_pos.quantile(Q_HIGH_MAIN)),
        "pos_q95": float(x_pos.quantile(Q_HIGH_WIDE)),
        "pos_min": float(x_pos.min()),
        "pos_max": float(x_pos.max()),
        "all_med": float(x_all.median()),
    }
    numeric_rules.append(row)

numeric_rules_df = pd.DataFrame(numeric_rules).sort_values("feature")
numeric_rules_path = os.path.join(OUT_DIR, "numeric_high_region_rules.csv")
numeric_rules_df.to_csv(numeric_rules_path, index=False, encoding="utf-8-sig")
print("[Saved]", numeric_rules_path)

def plot_numeric_distribution(df, col, out_prefix):
    x = _to_num(df[col])
    y = df[y_col].astype(int)
    dd = pd.DataFrame({col: x, "y": y}).dropna()
    if dd.shape[0] < 10:
        return

    fig, ax = plt.subplots(figsize=(8.6, 5.8))
    if USE_SEABORN:
        sns.histplot(data=dd, x=col, hue="y", kde=True, stat="density",
                     common_norm=False, bins=30, alpha=0.40, ax=ax)
    else:
        #  matplotlib:
        ax.hist(dd.loc[dd["y"] == 0, col], bins=30, density=True, alpha=0.40, label="y=0")
        ax.hist(dd.loc[dd["y"] == 1, col], bins=30, density=True, alpha=0.40, label="y=1")
        ax.legend()

    #  y=1  q10-q90 () q05-q95()
    pos = dd.loc[dd["y"] == 1, col]
    if len(pos) >= 5:
        q10, q90 = float(pos.quantile(Q_LOW_MAIN)), float(pos.quantile(Q_HIGH_MAIN))
        q05, q95 = float(pos.quantile(Q_LOW_WIDE)), float(pos.quantile(Q_HIGH_WIDE))
        ax.axvline(q10, linestyle="--", linewidth=1.5)
        ax.axvline(q90, linestyle="--", linewidth=1.5)
        ax.axvspan(q10, q90, alpha=0.12)
        ax.axvline(q05, linestyle=":", linewidth=1.2)
        ax.axvline(q95, linestyle=":", linewidth=1.2)

    ax.set_title(f"{col} distribution(y)")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    fig.tight_layout()

    fig.savefig(out_prefix + ".png", dpi=600, bbox_inches="tight")
    fig.savefig(out_prefix + ".svg", bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)

for col in num_cols_use + (["Size_nm"] if "Size_nm" in df_plot.columns else []):
    plot_numeric_distribution(df_plot, col, os.path.join(OUT_DIR, f"dist_{col.replace(' ', '_')}"))

print("[OK] Continuous featuresdistribution plotOutputs:", OUT_DIR)


# =========================
# 7) Two-dimensional continuous-featurescatter(hue=y)+ high-delivery region box(y=1)
# =========================
pairs = []
base_num = [c for c in ["Size", "Zeta Potential", "Admin"] if c in df_plot.columns]
for i in range(len(base_num)):
    for j in range(i + 1, len(base_num)):
        pairs.append((base_num[i], base_num[j]))

def plot_scatter_2d_with_pos_box(df, xcol, ycol2, out_prefix):
    dd = df[[xcol, ycol2, y_col]].copy()
    dd[xcol] = _to_num(dd[xcol])
    dd[ycol2] = _to_num(dd[ycol2])
    dd = dd.dropna()
    if dd.shape[0] < 10:
        return

    fig, ax = plt.subplots(figsize=(8.8, 6.4))
    if USE_SEABORN:
        sns.scatterplot(
            data=dd, x=xcol, y=ycol2, hue=y_col,
            alpha=0.75, s=55, edgecolor="white", linewidth=0.4,
            ax=ax
)
        ax.legend(title="y", loc="best")
    else:
        m0 = dd[y_col] == 0
        m1 = dd[y_col] == 1
        ax.scatter(dd.loc[m0, xcol], dd.loc[m0, ycol2], alpha=0.65, s=35, label="y=0")
        ax.scatter(dd.loc[m1, xcol], dd.loc[m1, ycol2], alpha=0.75, s=45, label="y=1")
        ax.legend()

    # high-delivery region box: y=1  q10-q90
    pos = dd.loc[dd[y_col] == 1]
    if pos.shape[0] >= 5:
        x_lo, x_hi = float(pos[xcol].quantile(Q_LOW_MAIN)), float(pos[xcol].quantile(Q_HIGH_MAIN))
        y_lo, y_hi = float(pos[ycol2].quantile(Q_LOW_MAIN)), float(pos[ycol2].quantile(Q_HIGH_MAIN))

        ax.axvline(x_lo, linestyle="--", linewidth=1.3)
        ax.axvline(x_hi, linestyle="--", linewidth=1.3)
        ax.axhline(y_lo, linestyle="--", linewidth=1.3)
        ax.axhline(y_hi, linestyle="--", linewidth=1.3)


        ax.fill_between([x_lo, x_hi], y_lo, y_hi, alpha=0.10)

        ax.text(
            x_lo, y_hi,
            f"pos q10-q90 box\nx:[{x_lo:.2f},{x_hi:.2f}] y:[{y_lo:.2f},{y_hi:.2f}]",
            fontsize=10,
            va="bottom",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="gray", lw=1)
)

    ax.set_title(f"{xcol} vs {ycol2}(y)")
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol2)
    ax.grid(True, linestyle="--", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()

    fig.savefig(out_prefix + ".png", dpi=600, bbox_inches="tight")
    fig.savefig(out_prefix + ".svg", bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)

for xcol, ycol2 in pairs:
    plot_scatter_2d_with_pos_box(
        df_plot, xcol, ycol2,
        os.path.join(OUT_DIR, f"scatter_{xcol.replace(' ', '_')}__{ycol2.replace(' ', '_')}")
)

print("[OK] scatterOutputs:", OUT_DIR)


# =========================
# 8):pos_rate + lift (CSV)+ TopLift
# =========================
overall_pos_rate = float(df_plot[y_col].mean())
cat_summary_records = []
recommended_categories = {}

def _safe_str(x):
    if pd.isna(x):
        return "<NA>"
    return str(x).strip()

for c in [x for x in CAT_COLS if x in df_plot.columns]:
    s = df_plot[c].map(_safe_str)
    y = df_plot[y_col].astype(int)

    tab = (
        pd.DataFrame({c: s, "y": y})
.groupby(c, dropna=False)["y"]
.agg(n="count", pos="sum", pos_rate="mean")
.reset_index()
)
    tab["lift"] = tab["pos_rate"] / overall_pos_rate if overall_pos_rate > 0 else np.nan
    tab = tab.sort_values(["lift", "pos_rate", "n"], ascending=False)

    out_csv = os.path.join(OUT_DIR, f"cat_posrate_lift_{c}.csv")
    tab.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # recommended category set:n>=CAT_MIN_COUNT  lift>=REC_LIFT_MIN
    rec = tab[(tab["n"] >= CAT_MIN_COUNT) & (tab["lift"] >= REC_LIFT_MIN)].copy()
    recommended_categories[c] = rec[c].astype(str).tolist()

    # ()
    for _, r in tab.iterrows():
        cat_summary_records.append({
            "feature": c,
            "category": r[c],
            "n": int(r["n"]),
            "pos": int(r["pos"]),
            "pos_rate": float(r["pos_rate"]),
            "lift": float(r["lift"]),
        })

    # TopLift (Top 15)
    top = tab[tab["n"] >= CAT_MIN_COUNT].head(15).copy()
    if top.shape[0] > 0:
        fig, ax = plt.subplots(figsize=(9.6, 6.2))
        ax.barh(top[c].astype(str)[:-1], top["lift"][:-1])
        ax.axvline(1.0, linestyle="--", linewidth=1.2)
        ax.set_title(f"{c}  Lift(Top {top.shape[0]},n>={CAT_MIN_COUNT})")
        ax.set_xlabel("Lift = pos_rate / overall_pos_rate")
        ax.set_ylabel(c)
        ax.grid(axis="x", linestyle="--", alpha=0.25)
        fig.tight_layout()

        fig.savefig(os.path.join(OUT_DIR, f"lift_bar_{c}.png"), dpi=600, bbox_inches="tight")
        fig.savefig(os.path.join(OUT_DIR, f"lift_bar_{c}.svg"), bbox_inches="tight")
        if SHOW_PLOTS:
            plt.show()
        plt.close(fig)

cat_summary_df = pd.DataFrame(cat_summary_records)
cat_summary_path = os.path.join(OUT_DIR, "categorical_posrate_lift_ALL.csv")
cat_summary_df.to_csv(cat_summary_path, index=False, encoding="utf-8-sig")
print("[Saved]", cat_summary_path)

rec_path = os.path.join(OUT_DIR, "recommended_categories_by_lift.json")
with open(rec_path, "w", encoding="utf-8") as f:
    json.dump(recommended_categories, f, ensure_ascii=False, indent=2)
print("[Saved]", rec_path)


# =========================
# 9) export search_space.json()
# =========================
search_space = {
    "target_de": TARGET_DE,
    "thr": float(thr),
    "thr_source": thr_src,
    "label_used": str(df_plot["_y_src_"].iloc[0]),
    "plot_df": df_plot_name,
    "overall_pos_rate": overall_pos_rate,
    "numeric_high_region": {},
    "categorical_recommended_by_lift": recommended_categories,
    "notes": {
        "numeric_region_definition": f"pos(y=1) quantile boxes: main[{Q_LOW_MAIN},{Q_HIGH_MAIN}], wide[{Q_LOW_WIDE},{Q_HIGH_WIDE}]",
        "categorical_rule": f"recommended if (n>={CAT_MIN_COUNT} and lift>={REC_LIFT_MIN})",
        "important": "used formodel input,prefer(for example Size=log10(nm)), Size_nm."
    }
}

for _, r in numeric_rules_df.iterrows():
    feat = str(r["feature"])
    search_space["numeric_high_region"][feat] = {
        "pos_q10": float(r["pos_q10"]),
        "pos_q90": float(r["pos_q90"]),
        "pos_q05": float(r["pos_q05"]),
        "pos_q95": float(r["pos_q95"]),
        "pos_med": float(r["pos_med"]),
        "pos_min": float(r["pos_min"]),
        "pos_max": float(r["pos_max"]),
    }

space_path = os.path.join(OUT_DIR, "search_space.json")
with open(space_path, "w", encoding="utf-8") as f:
    json.dump(search_space, f, ensure_ascii=False, indent=2)
print("[Saved]", space_path)


# =========================
# 10) ()
# =========================
print("\n==================== Summary ====================")
print("OUT_DIR =", OUT_DIR)
print("thr =", thr, "| thr_src =", thr_src)
print("pos_rate =", overall_pos_rate)
print("\n[Numeric high region rules](for sampling)")
print(numeric_rules_df[["feature", "pos_q10", "pos_q90", "pos_q05", "pos_q95"]].to_string(index=False))

print("\n[Recommended categories](categories above the lift threshold)")
for k, v in recommended_categories.items():
    print(f"- {k}: {v[:12]}{'.' if len(v)>12 else ''}")

print("\nNext: use search_space.json ranges and category sets for formulation generation and Top-N model screening.")


# Candidate ranking

Generate virtual candidates, score them with the selected model, and rank prioritized formulations.


In [ ]:
# =================== Random formulation generation, preprocessor reuse, saved-model scoring, and Top-N screening ===================
#  Notes(Default conservative screening settings):
# 1) Do not enable every filter by default:Default configuration "pairwise feasibility + quantile OOD + automatic numeric feasibility ranges"
# 2) Automatically use df_train [1%, 99%] generate FEAS_ABS_RANGES()
# 3) Keep Top-N explanation export, preprocessor saving, and filter-report metadata.

# Dependencies:df_train(required),recommended preprocessor(fit),and saved_models  payload/model
# Outputs:SAVE_DIR/FORMULA_GENERATION/  candidates_scored,topN,meta,TopNCSV

import os
import json
import glob
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

# --------- optional:catboost used for.cbm ---------
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None


# =========================================================
# 0) Feature switches with recommended defaults
# =========================================================

# --- (D) Save preprocessor; strongly recommended. ---
SAVE_PREPROCESSOR = True
PREPROCESSOR_SAVE_BASENAME = "preprocessor"
# --- (A) feasibility: ---
APPLY_FEASIBILITY_FILTERS = True

# (A1) numeric:"automatic generation"(train quantile)
# (overrides automatic ranges),ranges are defined in the model input space:
# for example Size If the model input uses log10(nm),the range should also use log10(nm).
FEAS_ABS_RANGES = {}  # leave empty=automatic generation;manual values can be provided {"Size":(0.0,3.5),.}

# (1%~99%)
AUTO_FEAS_Q_LOW = 0.01
AUTO_FEAS_Q_HIGH = 0.99

# (A2) numeric / (optional)
FEAS_NONNEGATIVE_COLS = ["Admin"]   #:dose is non-negative;set to [] if uncertain []
FEAS_INTEGER_COLS = []              # for example ["Admin"]  Admin

# (A3) Categorical tuple constraint; very strict and disabled by default.
FEAS_REQUIRE_SEEN_CAT_TUPLE = False

# (A4) Categorical pairwise constraint; recommended as a robust default.
FEAS_REQUIRE_SEEN_CAT_PAIRWISE = True

# --- (B) OOD:(quantile,) ---
APPLY_OOD_FILTERS = True

# (B1) quantile OOD:
OOD_USE_QUANTILE_BOUNDS = True
OOD_Q_LOW = 0.01
OOD_Q_HIGH = 0.99
OOD_QUANTILE_SOURCE = "train"     # "train"  "high"

# (B2) Mahalanobis distance OOD(stricter and disabled by default)
OOD_USE_MAHALANOBIS = False
OOD_MD_PCTL = 99.0                # percentile threshold from the training-set Mahalanobis-distance distribution

# --- (C) Top-N () ---
EXPORT_TOPN_EXPLANATION = True
TOPN_EXPLAIN_K = 200              # used for explanation Top-K(TOP_N)

# --- Candidate-generation scale ---
N_CANDIDATES = 50000
TOP_N = 200
RANDOM_STATE = 42

# Continuous-feature sampling
CONT_SAMPLING = "empirical"        # "empirical" / "uniform_q10_q90"
Q_LOW, Q_HIGH = 0.10, 0.90         # uniform_q10_q90


CAT_SAMPLING  = "train_empirical"  # "train_empirical" / "high_empirical" / "train_uniform"

#  round()
ROUND_CONT_DECIMALS = 3


# =========================================================
# 1):df_train / preprocessor /
# =========================================================
if "df_train" not in globals() or globals()["df_train"] is None:
    raise NameError(" df_train:(df_train).")
df_train = globals()["df_train"].copy()

TARGET_DE = globals().get("TARGET_DE", "DE_tumor")
if TARGET_DE not in df_train.columns:
    raise NameError(f"df_train  {TARGET_DE}.")

SAVE_DIR = globals().get("SAVE_DIR", "./output")
os.makedirs(SAVE_DIR, exist_ok=True)

SAVED_MODELS_DIR = os.path.join(SAVE_DIR, "saved_models")
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

USE_INTERACTIONS = bool(globals().get("USE_INTERACTIONS", False))

# ""(base_features)
base_features = globals().get(
    "base_features",
    ["Type", "MAT", "TS", "CT", "TM", "Shape", "Size", "Zeta Potential", "Admin"]
)
base_features = [c for c in base_features if c in df_train.columns]
if len(base_features) == 0:
    raise RuntimeError("base_features: df_train.")

NUM_COLS_USE = [c for c in ["Size", "Zeta Potential", "Admin"] if c in df_train.columns]
CAT_COLS_USE = [c for c in ["Type", "MAT", "TS", "CT", "TM", "Shape"] if c in df_train.columns]

print("[SAVE_DIR]", SAVE_DIR)
print("[TARGET_DE]", TARGET_DE)
print("[USE_INTERACTIONS]", USE_INTERACTIONS)
print("[base_features]", base_features)
print("[NUM_COLS_USE]", NUM_COLS_USE)
print("[CAT_COLS_USE]", CAT_COLS_USE)


# =========================================================
# 2) thr:high-deliverythreshold(DE threshold,threshold final_thr)
#    prefer thr_real / thr; train  0.75
# =========================================================
if "thr_real" in globals() and globals()["thr_real"] is not None:
    thr = float(globals()["thr_real"])
    thr_src = "thr_real()"
elif "thr" in globals() and globals()["thr"] is not None:
    thr = float(globals()["thr"])
    thr_src = "thr()"
else:
    thr = float(pd.to_numeric(df_train[TARGET_DE], errors="coerce").quantile(0.75))
    thr_src = "df_train Q0.75()"

df_train["_y_high_"] = (pd.to_numeric(df_train[TARGET_DE], errors="coerce") >= thr).astype(int)
df_high = df_train[df_train["_y_high_"] == 1].copy()

print(f"[thr] {thr:.6f} | ={thr_src}")
print(f"[Train] n={len(df_train)} | high_n={len(df_high)} | pos_rate={df_train['_y_high_'].mean():.3f}")
if len(df_high) < 10:
    raise ValueError("high-delivery(high_n<10),thr.")


# =========================================================
# 3) Preprocessing:prefer globals.preprocessor(fit)
# =========================================================
if "preprocessor" in globals() and globals()["preprocessor"] is not None:
    preprocessor = globals()["preprocessor"]
    pre_src = "globals.preprocessor"
else:
    cand_paths = sorted(glob.glob(os.path.join(SAVED_MODELS_DIR, "*preprocessor*.joblib")))
    if len(cand_paths) == 0:
        raise RuntimeError(
            " preprocessor(globals,).\n"
            ": joblib.dump(preprocessor, '.preprocessor.joblib'), load."
)
    preprocessor = joblib.load(cand_paths[-1])
    pre_src = f"joblib({cand_paths[-1]})"

print("[Preprocessor]", pre_src)

# --- (D) save preprocessor ---
preprocessor_path_saved = None
if SAVE_PREPROCESSOR:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    preprocessor_path_saved = os.path.join(
        SAVED_MODELS_DIR,
        f"{PREPROCESSOR_SAVE_BASENAME}_{TARGET_DE}_inter{int(USE_INTERACTIONS)}_{ts}.joblib"
)
    try:
        joblib.dump(preprocessor, preprocessor_path_saved)
        print("[Saved Preprocessor]", preprocessor_path_saved)
    except Exception as e:
        print("[Warn] preprocessor save:", repr(e))


# =========================================================
# 4):prefer best_model/best_cat; saved_models/*_payload.joblib.cbm
# =========================================================
def _pick_model_in_memory():
    """
     4:
      model, model_src, payload_disk, payload_path_disk
    """
    if "best_model" in globals() and globals()["best_model"] is not None:
        return globals()["best_model"], "globals.best_model", None, None
    if "best_cat" in globals() and globals()["best_cat"] is not None:
        return globals()["best_cat"], "globals.best_cat", None, None
    for nm in ["best_xgb", "best_lgbm", "best_rf", "best_svm", "best_LogisticRegression"]:
        if nm in globals() and globals()[nm] is not None:
            return globals()[nm], f"globals.{nm}", None, None
    return None, None, None, None

def _load_payload_and_model_from_disk():
    payloads = (
        sorted(glob.glob(os.path.join(SAVED_MODELS_DIR, "*_payload.joblib"))) +
        sorted(glob.glob(os.path.join(SAVED_MODELS_DIR, "*payload*.joblib")))
)
    if len(payloads) == 0:
        return None, None, None, None

    payload_path = payloads[-1]
    payload = joblib.load(payload_path)
    if not isinstance(payload, dict):
        return None, None, None, None

    model_path = payload.get("model_path", None)
    if model_path is None:
        return None, None, payload, payload_path


    if not os.path.isabs(model_path):
        cand1 = os.path.join(SAVE_DIR, model_path)
        cand2 = os.path.join(SAVED_MODELS_DIR, model_path)
        if os.path.exists(cand1):
            model_path = cand1
        elif os.path.exists(cand2):
            model_path = cand2

    if not os.path.exists(model_path):
        return None, None, payload, payload_path

    # CatBoost.cbm
    if str(model_path).lower().endswith(".cbm"):
        if CatBoostClassifier is None:
            raise RuntimeError(".cbm, import catboost./ catboost.")
        m = CatBoostClassifier()
        m.load_model(model_path)
        return m, f"CatBoost.load_model({model_path})", payload, payload_path

    #:joblib
    m = joblib.load(model_path)
    return m, f"joblib({model_path})", payload, payload_path

model, model_src, payload_disk, payload_path_disk = _pick_model_in_memory()
if model is None:
    model, model_src, payload_disk, payload_path_disk = _load_payload_and_model_from_disk()

if model is None:
    raise RuntimeError(":save(best_model/best_cat  saved_models/*_payload.joblib +.cbm).")

print("[Model]", type(model), "|:", model_src)
if payload_disk is not None:
    print("[Payload]", payload_path_disk)
    if "final_thr" in payload_disk:
        print("[Payload final_thr]", payload_disk.get("final_thr"))


# =========================================================
# 5) candidate -> / -> preprocessor.transform
# =========================================================
def _to_num(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

def candidate_to_X(df_feat: pd.DataFrame):
    df0 = df_feat.copy()

    # Categorical columns:()
    for c in CAT_COLS_USE:
        if c in df0.columns:
            df0[c] = df0[c].fillna("<NA>").astype(str)

    # Numeric columns: numeric
    for c in NUM_COLS_USE:
        if c in df0.columns:
            df0[c] = _to_num(df0[c])

    # /()
    if "normalize_categorical_df" in globals() and callable(globals()["normalize_categorical_df"]):
        df0 = globals()["normalize_categorical_df"](df0, ["Type", "MAT", "TS", "CT", "TM", "Shape"])
    if "apply_semantic_mapping" in globals() and callable(globals()["apply_semantic_mapping"]):
        df0 = globals()["apply_semantic_mapping"](df0)

    # interaction features()
    if USE_INTERACTIONS:
        if "add_interactions" not in globals() or not callable(globals()["add_interactions"]):
            raise RuntimeError("USE_INTERACTIONS=True  add_interactions.")
        df0 = globals()["add_interactions"](df0)

    #: fit  preprocessor.transform
    X = preprocessor.transform(df0)
    return X


# =========================================================
# 6) generatecandidate: df_high;
# =========================================================
rng = np.random.default_rng(RANDOM_STATE)

def sample_continuous_from_high(df_high_, col, n, mode="empirical"):
    x = _to_num(df_high_[col]).dropna()
    if len(x) < 5:
        x = _to_num(df_train[col]).dropna()
    if len(x) < 5:
        raise ValueError(f" {col},.")

    if mode == "empirical":
        out = rng.choice(x.values, size=n, replace=True)
    elif mode == "uniform_q10_q90":
        lo, hi = float(x.quantile(Q_LOW)), float(x.quantile(Q_HIGH))
        if hi <= lo:
            out = rng.choice(x.values, size=n, replace=True)
        else:
            out = rng.uniform(lo, hi, size=n)
    else:
        raise ValueError("CONT_SAMPLING  empirical / uniform_q10_q90")

    if ROUND_CONT_DECIMALS is not None:
        out = np.round(out.astype(float), ROUND_CONT_DECIMALS)
    return out

def sample_categorical_indep(df_source_, col, n, mode="empirical"):
    s = df_source_[col].fillna("<NA>").astype(str)
    vc = s.value_counts()
    cats = vc.index.tolist()
    if len(cats) == 0:
        raise ValueError(f" {col} category.")

    if mode == "empirical":
        probs = (vc.values / vc.values.sum()).astype(float)
        return rng.choice(cats, size=n, replace=True, p=probs)
    elif mode == "uniform":
        return rng.choice(cats, size=n, replace=True)
    else:
        raise ValueError("mode  empirical / uniform")


if CAT_SAMPLING == "high_empirical":
    cat_source = df_high
    cat_mode = "empirical"
elif CAT_SAMPLING == "train_empirical":
    cat_source = df_train
    cat_mode = "empirical"
elif CAT_SAMPLING == "train_uniform":
    cat_source = df_train
    cat_mode = "uniform"
else:
    raise ValueError("CAT_SAMPLING  train_empirical / high_empirical / train_uniform")

# candidate
candidates = pd.DataFrame(index=np.arange(N_CANDIDATES))

for c in NUM_COLS_USE:
    candidates[c] = sample_continuous_from_high(df_high, c, N_CANDIDATES, mode=CONT_SAMPLING)

for c in CAT_COLS_USE:
    candidates[c] = sample_categorical_indep(cat_source, c, N_CANDIDATES, mode=cat_mode)

# ()
dedup_cols = [c for c in (NUM_COLS_USE + CAT_COLS_USE) if c in candidates.columns]
before_dedup = len(candidates)
candidates = candidates.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
print(f"[Candidates] requested={before_dedup} | after_dedup={len(candidates)}")


# =========================================================
# 7) (A) feasibility + (B) OOD (,)
# =========================================================
def _safe_str(x):
    if pd.isna(x):
        return "<NA>"
    return str(x).strip()

def build_train_combo_sets(df_ref: pd.DataFrame):
    """training setcategorytuple + pairwise,used forfeasibility."""
    sets = {}
    if len(CAT_COLS_USE) > 0:
        cat_tuple_set = set(
            tuple(_safe_str(v) for v in row)
            for row in df_ref[CAT_COLS_USE].fillna("<NA>").astype(str).itertuples(index=False, name=None)
)
        sets["cat_tuple_set"] = cat_tuple_set

        pair_sets = {}
        for i in range(len(CAT_COLS_USE)):
            for j in range(i + 1, len(CAT_COLS_USE)):
                a, b = CAT_COLS_USE[i], CAT_COLS_USE[j]
                pair_sets[(a, b)] = set(
                    (va, vb)
                    for va, vb in df_ref[[a, b]].fillna("<NA>").astype(str).itertuples(index=False, name=None)
)
        sets["cat_pair_sets"] = pair_sets
    return sets

train_combo_sets = build_train_combo_sets(df_train)

def compute_numeric_quantile_bounds(df_ref: pd.DataFrame, cols, q_low, q_high):
    bounds = {}
    for c in cols:
        if c not in df_ref.columns:
            continue
        x = _to_num(df_ref[c]).dropna()
        if len(x) < 10:
            continue
        bounds[c] = (float(x.quantile(q_low)), float(x.quantile(q_high)))
    return bounds

def mahalanobis_setup(df_ref: pd.DataFrame, cols):
    """training setMahalanobis distanceparameters; (mean, inv_cov, md_threshold, ok_flag)."""
    cols = [c for c in cols if c in df_ref.columns]
    if len(cols) < 2:
        return None, None, None, False

    X = df_ref[cols].copy()
    for c in cols:
        X[c] = _to_num(X[c])
    X = X.dropna()
    if X.shape[0] < 20:
        return None, None, None, False

    M = X.to_numpy(dtype=float)
    mu = M.mean(axis=0)
    cov = np.cov(M, rowvar=False)

    ridge = 1e-8 * np.eye(cov.shape[0])
    cov = cov + ridge
    try:
        inv_cov = np.linalg.inv(cov)
    except Exception:
        return None, None, None, False

    dif = M - mu
    md2 = np.einsum("ij,jk,ik->i", dif, inv_cov, dif)
    md = np.sqrt(np.maximum(md2, 0.0))
    thr_md = float(np.percentile(md, OOD_MD_PCTL))
    return mu, inv_cov, thr_md, True

def mahalanobis_distance(M, mu, inv_cov):
    dif = M - mu
    md2 = np.einsum("ij,jk,ik->i", dif, inv_cov, dif)
    return np.sqrt(np.maximum(md2, 0.0))

# ---------- automatic generation FEAS_ABS_RANGES() ----------
if APPLY_FEASIBILITY_FILTERS:
    if FEAS_ABS_RANGES is None:
        FEAS_ABS_RANGES = {}

    #, train  [AUTO_FEAS_Q_LOW, AUTO_FEAS_Q_HIGH]
    if isinstance(FEAS_ABS_RANGES, dict) and len(FEAS_ABS_RANGES) == 0:
        auto_ranges = compute_numeric_quantile_bounds(df_train, NUM_COLS_USE, AUTO_FEAS_Q_LOW, AUTO_FEAS_Q_HIGH)
        FEAS_ABS_RANGES.update(auto_ranges)
        print(f"[Auto FEAS_ABS_RANGES] from train q[{AUTO_FEAS_Q_LOW},{AUTO_FEAS_Q_HIGH}] ->", FEAS_ABS_RANGES)
    else:
        print("[FEAS_ABS_RANGES] (manual/partial) ->", FEAS_ABS_RANGES)

# ---------- OOD quantile bounds ----------
quantile_bounds = None
if APPLY_OOD_FILTERS and OOD_USE_QUANTILE_BOUNDS:
    ref = df_train if OOD_QUANTILE_SOURCE == "train" else df_high
    quantile_bounds = compute_numeric_quantile_bounds(ref, NUM_COLS_USE, OOD_Q_LOW, OOD_Q_HIGH)
    print(f"[OOD quantile bounds] source={OOD_QUANTILE_SOURCE} q[{OOD_Q_LOW},{OOD_Q_HIGH}] ->", quantile_bounds)

# ---------- Mahalanobis setup ----------
md_mu = md_inv = md_thr = None
md_ok = False
if APPLY_OOD_FILTERS and OOD_USE_MAHALANOBIS:
    md_mu, md_inv, md_thr, md_ok = mahalanobis_setup(df_train, NUM_COLS_USE)
    print("[OOD mahalanobis] enabled:", md_ok, "| md_thr:", md_thr)

def apply_filters(df_cand: pd.DataFrame):
    dfc = df_cand.copy()
    mask = np.ones(len(dfc), dtype=bool)
    report = {"n_in": int(len(dfc)), "dropped": {}}

    # ---------- (A) feasibility:numeric ----------
    if APPLY_FEASIBILITY_FILTERS and isinstance(FEAS_ABS_RANGES, dict) and len(FEAS_ABS_RANGES) > 0:
        for col, (lo, hi) in FEAS_ABS_RANGES.items():
            if col not in dfc.columns:
                continue
            x = _to_num(dfc[col])
            m = x.notna() & (x >= float(lo)) & (x <= float(hi))
            before = int(mask.sum())
            mask &= m.to_numpy()
            report["dropped"][f"feas_abs_range_{col}"] = int(before - mask.sum())

    # ---------- (A) feasibility: ----------
    if APPLY_FEASIBILITY_FILTERS and FEAS_NONNEGATIVE_COLS:
        for col in FEAS_NONNEGATIVE_COLS:
            if col not in dfc.columns:
                continue
            x = _to_num(dfc[col])
            m = x.notna() & (x >= 0)
            before = int(mask.sum())
            mask &= m.to_numpy()
            report["dropped"][f"feas_nonneg_{col}"] = int(before - mask.sum())

    # ---------- (A) feasibility: ----------
    if APPLY_FEASIBILITY_FILTERS and FEAS_INTEGER_COLS:
        for col in FEAS_INTEGER_COLS:
            if col not in dfc.columns:
                continue
            x = _to_num(dfc[col])
            m = x.notna() & np.isclose(x % 1, 0)
            before = int(mask.sum())
            mask &= m.to_numpy()
            report["dropped"][f"feas_integer_{col}"] = int(before - mask.sum())

    # ---------- (A) feasibility:categorytuple ----------
    if APPLY_FEASIBILITY_FILTERS and FEAS_REQUIRE_SEEN_CAT_TUPLE and len(CAT_COLS_USE) > 0:
        cat_tuple_set = train_combo_sets.get("cat_tuple_set", set())
        tuples = [
            tuple(_safe_str(v) for v in row)
            for row in dfc[CAT_COLS_USE].fillna("<NA>").astype(str).itertuples(index=False, name=None)
]
        m = np.array([t in cat_tuple_set for t in tuples], dtype=bool)
        before = int(mask.sum())
        mask &= m
        report["dropped"]["feas_seen_cat_tuple"] = int(before - mask.sum())

    # ---------- (A) feasibility:categorypairwise ----------
    if APPLY_FEASIBILITY_FILTERS and FEAS_REQUIRE_SEEN_CAT_PAIRWISE and len(CAT_COLS_USE) > 1:
        pair_sets = train_combo_sets.get("cat_pair_sets", {})
        for (a, b), allowed in pair_sets.items():
            va = dfc[a].fillna("<NA>").astype(str).to_numpy()
            vb = dfc[b].fillna("<NA>").astype(str).to_numpy()
            m = np.array([(x, y) in allowed for x, y in zip(va, vb)], dtype=bool)
            before = int(mask.sum())
            mask &= m
            report["dropped"][f"feas_seen_pair_{a}__{b}"] = int(before - mask.sum())

    # ---------- (B) OOD:quantile ----------
    if APPLY_OOD_FILTERS and OOD_USE_QUANTILE_BOUNDS and quantile_bounds:
        for col, (lo, hi) in quantile_bounds.items():
            x = _to_num(dfc[col])
            m = x.notna() & (x >= lo) & (x <= hi)
            before = int(mask.sum())
            mask &= m.to_numpy()
            report["dropped"][f"ood_quantile_{col}[{OOD_Q_LOW},{OOD_Q_HIGH}]"] = int(before - mask.sum())

    # ---------- (B) OOD:Mahalanobis distance ----------
    if APPLY_OOD_FILTERS and OOD_USE_MAHALANOBIS and md_ok:
        Xn = dfc[NUM_COLS_USE].copy()
        for c in NUM_COLS_USE:
            Xn[c] = _to_num(Xn[c])
        Xn = Xn.to_numpy(dtype=float)

        nan_mask = np.any(~np.isfinite(Xn), axis=1)
        md = np.full(len(dfc), np.inf, dtype=float)
        ok_rows = ~nan_mask
        if ok_rows.any():
            md[ok_rows] = mahalanobis_distance(Xn[ok_rows], md_mu, md_inv)

        m = (md <= md_thr)
        before = int(mask.sum())
        mask &= m
        report["dropped"][f"ood_mahalanobis<=P{OOD_MD_PCTL}({md_thr:.4f})"] = int(before - mask.sum())
        dfc["mahalanobis_dist"] = md

    df_out = dfc.loc[mask].reset_index(drop=True)
    report["n_out"] = int(len(df_out))
    return df_out, report

candidates, filter_report = apply_filters(candidates)
print("[Candidates after filters]", candidates.shape)
print("[Filter report]", json.dumps(filter_report, ensure_ascii=False, indent=2))

if len(candidates) == 0:
    raise RuntimeError(
        " candidates=0.\n"
        ":\n"
        "1)  FEAS_REQUIRE_SEEN_CAT_TUPLE()\n"
        "2)  OOD_USE_MAHALANOBIS()\n"
        "3)  OOD_Q_LOW/OOD_Q_HIGH(for example 0.005~0.995)\n"
        "4)  N_CANDIDATES(for example 200000)"
)


# =========================================================
# 8) candidate -> Preprocessing ->
# =========================================================
cand_feat = candidates[base_features].copy()
X_cand = candidate_to_X(cand_feat)

if hasattr(model, "predict_proba"):
    proba = model.predict_proba(X_cand)[:, 1]
elif hasattr(model, "decision_function"):
    z = model.decision_function(X_cand)
    proba = 1.0 / (1.0 + np.exp(-z))
else:
    raise RuntimeError(" predict_proba / decision_function,.")

candidates_scored = candidates.copy()
candidates_scored["pred_proba_high"] = np.asarray(proba, dtype=float)
candidates_scored = candidates_scored.sort_values("pred_proba_high", ascending=False).reset_index(drop=True)

# payload  final_thr(threshold,optional)
final_thr = None
if payload_disk is not None and isinstance(payload_disk, dict):
    if payload_disk.get("final_thr", None) is not None:
        try:
            final_thr = float(payload_disk["final_thr"])
        except Exception:
            final_thr = None


# =========================================================
# 9) saveOutputs(candidate + TopN + meta)
# =========================================================
OUT_DIR = os.path.join(SAVE_DIR, "FORMULA_GENERATION")
os.makedirs(OUT_DIR, exist_ok=True)

out_all = os.path.join(OUT_DIR, "generated_candidates_scored.csv")
out_top = os.path.join(OUT_DIR, f"generated_top{TOP_N}.csv")
candidates_scored.to_csv(out_all, index=False, encoding="utf-8-sig")
candidates_scored.head(TOP_N).to_csv(out_top, index=False, encoding="utf-8-sig")

if final_thr is not None:
    passed = candidates_scored[candidates_scored["pred_proba_high"] >= final_thr].copy()
    out_pass = os.path.join(OUT_DIR, f"generated_pass_final_thr_{final_thr:.3f}.csv")
    passed.to_csv(out_pass, index=False, encoding="utf-8-sig")
    print("[Saved] pass_final_thr:", out_pass, "| n=", len(passed))

meta = {
    "target_de": TARGET_DE,
    "thr": float(thr),
    "thr_source": thr_src,
    "high_definition": f"{TARGET_DE} >= thr (thr computed from TRAIN only)",
    "n_train": int(len(df_train)),
    "n_high": int(len(df_high)),
    "pos_rate_train": float(df_train["_y_high_"].mean()),
    "n_candidates_requested": int(N_CANDIDATES),

    "n_candidates_after_dedup": int(before_dedup if False else 0),  # ()
    "n_candidates_after_filters": int(len(candidates_scored)),
    "filter_report": filter_report,
    "sampling": {
        "cont_sampling": CONT_SAMPLING,
        "cat_sampling": CAT_SAMPLING,
        "q_low": Q_LOW,
        "q_high": Q_HIGH,
        "round_cont_decimals": ROUND_CONT_DECIMALS,
    },
    "features": {
        "base_features": base_features,
        "num_cols": NUM_COLS_USE,
        "cat_cols": CAT_COLS_USE,
        "use_interactions": USE_INTERACTIONS,
    },
    "model": {
        "model_source": model_src,
        "payload_path": payload_path_disk,
        "final_thr_from_payload": final_thr,
    },
    "preprocessor": {
        "preprocessor_source": pre_src,
        "preprocessor_saved_path": preprocessor_path_saved,
    },
    "filters": {
        "APPLY_FEASIBILITY_FILTERS": APPLY_FEASIBILITY_FILTERS,
        "FEAS_ABS_RANGES": FEAS_ABS_RANGES,
        "FEAS_NONNEGATIVE_COLS": FEAS_NONNEGATIVE_COLS,
        "FEAS_INTEGER_COLS": FEAS_INTEGER_COLS,
        "FEAS_REQUIRE_SEEN_CAT_TUPLE": FEAS_REQUIRE_SEEN_CAT_TUPLE,
        "FEAS_REQUIRE_SEEN_CAT_PAIRWISE": FEAS_REQUIRE_SEEN_CAT_PAIRWISE,
        "APPLY_OOD_FILTERS": APPLY_OOD_FILTERS,
        "OOD_USE_QUANTILE_BOUNDS": OOD_USE_QUANTILE_BOUNDS,
        "OOD_Q_LOW": OOD_Q_LOW,
        "OOD_Q_HIGH": OOD_Q_HIGH,
        "OOD_QUANTILE_SOURCE": OOD_QUANTILE_SOURCE,
        "OOD_USE_MAHALANOBIS": OOD_USE_MAHALANOBIS,
        "OOD_MD_PCTL": OOD_MD_PCTL,
    },
    "notes": {
        "no_leakage": "thr// df_train  df_high(df_train), test.",
        "recommended_mode": "Default configuration pairwise feasibility + quantile OOD(/tuple,)."
    }
}
out_meta = os.path.join(OUT_DIR, "generation_meta.json")
with open(out_meta, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("\n[Saved]")
print(" - all:", out_all)
print(" - top:", out_top)
print(" - meta:", out_meta)

print("\nTop5 preview:")
print(candidates_scored.head(5))


# =========================================================
# 10) (C) Top-N:category /  / Continuous features(export CSV)
# =========================================================
def _freq_table(series: pd.Series):
    vc = series.value_counts(dropna=False)
    out = pd.DataFrame({"value": vc.index.astype(str), "count": vc.values})
    out["freq"] = out["count"] / float(out["count"].sum()) if out["count"].sum() > 0 else 0.0
    return out

def export_topn_explanations(df_scored: pd.DataFrame, df_train_ref: pd.DataFrame, out_dir: str, topk: int):
    topk = int(min(max(topk, 1), len(df_scored)))
    df_top = df_scored.head(topk).copy()
    df_all = df_scored.copy()

    # ---- Continuous features:TopK vs All ----
    num_rows = []
    for c in NUM_COLS_USE:
        if c not in df_all.columns:
            continue
        a = _to_num(df_all[c]).dropna()
        t = _to_num(df_top[c]).dropna()
        if len(a) < 10 or len(t) < 10:
            continue
        num_rows.append({
            "feature": c,
            "topk_n": int(len(t)),
            "all_n": int(len(a)),
            "topk_mean": float(t.mean()),
            "topk_med": float(t.median()),
            "topk_q10": float(t.quantile(0.10)),
            "topk_q90": float(t.quantile(0.90)),
            "all_mean": float(a.mean()),
            "all_med": float(a.median()),
            "all_q10": float(a.quantile(0.10)),
            "all_q90": float(a.quantile(0.90)),
        })
    num_df = pd.DataFrame(num_rows)
    num_path = os.path.join(out_dir, f"top{topk}_numeric_summary.csv")
    num_df.to_csv(num_path, index=False, encoding="utf-8-sig")

    # ---- Categorical features:TopK vs All vs Train ----
    for c in CAT_COLS_USE:
        if c not in df_all.columns:
            continue

        tab_top = _freq_table(df_top[c].fillna("<NA>").astype(str)).rename(
            columns={"count": "top_count", "freq": "top_freq"}
)
        tab_all = _freq_table(df_all[c].fillna("<NA>").astype(str)).rename(
            columns={"count": "all_count", "freq": "all_freq"}
)
        tab_tr = _freq_table(df_train_ref[c].fillna("<NA>").astype(str)).rename(
            columns={"count": "train_count", "freq": "train_freq"}
)

        m = tab_top.merge(tab_all, on="value", how="left").merge(tab_tr, on="value", how="left")
        m["enrich_vs_all"] = (m["top_freq"] / m["all_freq"]).replace([np.inf, -np.inf], np.nan)
        m["enrich_vs_train"] = (m["top_freq"] / m["train_freq"]).replace([np.inf, -np.inf], np.nan)
        m = m.sort_values(["enrich_vs_all", "top_count"], ascending=False)

        outp = os.path.join(out_dir, f"top{topk}_cat_enrichment_{c}.csv")
        m.to_csv(outp, index=False, encoding="utf-8-sig")

    # ----: key ----
    if len(CAT_COLS_USE) > 0:
        def make_key(df_):
            return df_[CAT_COLS_USE].fillna("<NA>").astype(str).agg("||".join, axis=1)

        df_top["_cat_combo_"] = make_key(df_top)
        df_all["_cat_combo_"] = make_key(df_all)
        df_tr = df_train_ref.copy()
        df_tr["_cat_combo_"] = make_key(df_tr)

        tab_top = _freq_table(df_top["_cat_combo_"]).rename(columns={"value": "combo", "count": "top_count", "freq": "top_freq"})
        tab_all = _freq_table(df_all["_cat_combo_"]).rename(columns={"value": "combo", "count": "all_count", "freq": "all_freq"})
        tab_tr  = _freq_table(df_tr["_cat_combo_"]).rename(columns={"value": "combo", "count": "train_count", "freq": "train_freq"})

        comb = tab_top.merge(tab_all, on="combo", how="left").merge(tab_tr, on="combo", how="left")
        comb["enrich_vs_all"] = (comb["top_freq"] / comb["all_freq"]).replace([np.inf, -np.inf], np.nan)
        comb["enrich_vs_train"] = (comb["top_freq"] / comb["train_freq"]).replace([np.inf, -np.inf], np.nan)
        comb = comb.sort_values(["enrich_vs_all", "top_count"], ascending=False)

        comb_path = os.path.join(out_dir, f"top{topk}_cat_combo_enrichment.csv")
        comb.to_csv(comb_path, index=False, encoding="utf-8-sig")
    else:
        comb_path = None

    return {
        "numeric_summary_csv": os.path.basename(num_path),
        "cat_enrichment_csvs": [f"top{topk}_cat_enrichment_{c}.csv" for c in CAT_COLS_USE],
        "cat_combo_enrichment_csv": (os.path.basename(comb_path) if comb_path else None),
    }

if EXPORT_TOPN_EXPLANATION:
    explain_k = int(min(TOPN_EXPLAIN_K, TOP_N, len(candidates_scored)))
    explain_files = export_topn_explanations(candidates_scored, df_train, OUT_DIR, topk=explain_k)
    explain_meta_path = os.path.join(OUT_DIR, f"top{explain_k}_explain_files.json")
    with open(explain_meta_path, "w", encoding="utf-8") as f:
        json.dump(explain_files, f, ensure_ascii=False, indent=2)
    print(f"\n[Top-N Explain Saved] topK={explain_k}")
    print(" -", explain_meta_path)

print("\n[Done] FORMULA_GENERATION finished.")


# Output export

The workflow exports scored virtual candidates, top-ranked candidate tables, cancer-type-specific outputs, and local working-range summaries under the repository output directory.
